In [1]:
import pandas as pd
import numpy as np
import subprocess
import os
from datetime import datetime, timedelta, time, date
import datetime
from datetime import datetime
import locale
import seaborn as sns
import matplotlib.pyplot as plt
import re

In [ ]:
dia = "20251015"

dia_i = '15/10/2025'
dia_f = '15/10/2025'

Rutas

In [3]:
# Ruta al archivo de Excel de Pir_rutas
file_path = 'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2024/Pir_rutas.xlsx'

# Leer la pestaña 'Kilometraje'
rutas = pd.read_excel(file_path)

rutas

,Tarea,Ruta,PIR,linea,ruta_sae
0,16-1 TIERRA GRATA_CIRCULAR,16-1,Portal Dorado,59,2631
1,16-1 TIERRA GRATA_FIN EN 201A05,16-1,Portal Dorado,59,2632
2,16-1 TIERRA GRATA_CIRCULAR,16-1 TIERRA,Portal Dorado,59,2631
3,16-1 TIERRA GRATA_FIN EN 201A05,16-1 TIERRA,Portal Dorado,59,2632
4,16-2 ENGATIVA CENTRO_CIRCULAR,16-2 ENG,Portal Dorado,60,2200
...,...,...,...,...,...
69,BD237,BD237,Verbena,1686,8342
70,128_CICLOVIA,128,Villa Gladys,1245,8381
71,DD212_FMS,DD212,Verbena,10305,10590
72,576_MIG_IDA_4,576.,Verbena,1208,8452


IPH bruto

In [4]:
#IPH GM

iph_zonal = pd.read_csv(f'Z:/01 base_datos/12 iph/{dia}_iph_zonal.csv')
iph_troncal = pd.read_csv(f'Z:/01 base_datos/34 iph alim/{dia}_iph_alim.csv')

#Concatenado de dataframes IPH Completa

iph1 = pd.concat([iph_zonal,iph_troncal], ignore_index=True)

iph1

,JornadaTipo,TipoDia,Operador,Instante,ServBus,Evento,Linea,Coche,Sublinea,Ruta,Punto,TipoNodo,Viaje,ServicioCondEnt,TurnoEnt,OperadorEnt,ServicioCondSal,TipoVehiculo
0,GC250624T2,CN00111830,105,3:20:00,CNLYU0001,18,1300,1,NaN,NaN,142,5.0,1,CE130330,1.0,105.0,NaN,8
1,GC250624T2,CN00111830,105,4:00:00,CNLYU0001,4,1300,1,NaN,NaN,52372,1.0,1,NaN,NaN,NaN,NaN,8
2,GC250624T2,CN00111830,105,4:00:00,CNLYU0001,11,1300,1,4097.0,5658.0,52372,1.0,2,NaN,NaN,NaN,NaN,8
3,GC250624T2,CN00111830,105,4:08:48,CNLYU0001,0,1300,1,4097.0,5658.0,52802,1.0,2,NaN,NaN,NaN,NaN,8
4,GC250624T2,CN00111830,105,4:17:36,CNLYU0001,0,1300,1,4097.0,5658.0,53222,1.0,2,NaN,NaN,NaN,NaN,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36316,GMAD250519,CE00101255,105,20:58:45,CE4E70039,4,60,12,1665.0,2200.0,82,1.0,17,NaN,NaN,NaN,NaN,4
36317,GMAD250519,CE00101255,105,20:58:45,CE4E70039,3,60,12,1665.0,2200.0,82,1.0,18,NaN,NaN,NaN,NaN,4
36318,GMAD250519,CE00101255,105,21:40:15,CE4E70039,12,60,12,1665.0,2200.0,82,1.0,18,NaN,NaN,NaN,NaN,4
36319,GMAD250519,CE00101255,105,21:40:15,CE4E70039,2,60,12,NaN,NaN,82,1.0,19,NaN,NaN,NaN,NaN,4


In [5]:
# Función para corregir tiempos con "24:00:00"
def fix_time(date_str):
    if '24:' in date_str:
        return date_str.replace('24:', '00:')
    elif '25:' in date_str:
        return date_str.replace('25:', '01:')
    elif '26:' in date_str:
        return date_str.replace('26:', '02:')
    elif '27:' in date_str:
        return date_str.replace('27:', '03:')
    elif '28:' in date_str:
        return date_str.replace('28:', '04:')
    elif '29:' in date_str:
        return date_str.replace('29:', '05:')
    else:
        return date_str

# Función para convertir tiempo a segundos
def time_to_seconds(time_obj):
    return time_obj.hour * 3600 + time_obj.minute * 60 + time_obj.second

# Convertir la columna de tiempo a cadenas
iph1['Instante'] = iph1['Instante'].astype(str)

# Aplicar la función para corregir los tiempos
iph1['Instante'] = iph1['Instante'].apply(fix_time)

# Convertir la columna de tiempo a datetime, usando errors='coerce' para manejar errores
iph1['Instante'] = pd.to_datetime(iph1['Instante'], format='%H:%M:%S', errors='coerce')

# Eliminar la fecha predeterminada para trabajar solo con la parte de tiempo
iph1['Instante'] = iph1['Instante'].dt.time

# Manejar NaT después de la conversión
iph1['Instante'] = iph1['Instante'].apply(lambda x: x if pd.notnull(x) else pd.Timestamp('00:00:00').time())

# Convertir la columna 'Instante' a segundos
iph1['Instante_seg'] = iph1['Instante'].apply(time_to_seconds).astype(int)

# Convertir la columna 'Instante' a segundos
iph1

,JornadaTipo,TipoDia,Operador,Instante,ServBus,Evento,Linea,Coche,Sublinea,Ruta,Punto,TipoNodo,Viaje,ServicioCondEnt,TurnoEnt,OperadorEnt,ServicioCondSal,TipoVehiculo,Instante_seg
0,GC250624T2,CN00111830,105,03:20:00,CNLYU0001,18,1300,1,NaN,NaN,142,5.0,1,CE130330,1.0,105.0,NaN,8,12000
1,GC250624T2,CN00111830,105,04:00:00,CNLYU0001,4,1300,1,NaN,NaN,52372,1.0,1,NaN,NaN,NaN,NaN,8,14400
2,GC250624T2,CN00111830,105,04:00:00,CNLYU0001,11,1300,1,4097.0,5658.0,52372,1.0,2,NaN,NaN,NaN,NaN,8,14400
3,GC250624T2,CN00111830,105,04:08:48,CNLYU0001,0,1300,1,4097.0,5658.0,52802,1.0,2,NaN,NaN,NaN,NaN,8,14928
4,GC250624T2,CN00111830,105,04:17:36,CNLYU0001,0,1300,1,4097.0,5658.0,53222,1.0,2,NaN,NaN,NaN,NaN,8,15456
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36316,GMAD250519,CE00101255,105,20:58:45,CE4E70039,4,60,12,1665.0,2200.0,82,1.0,17,NaN,NaN,NaN,NaN,4,75525
36317,GMAD250519,CE00101255,105,20:58:45,CE4E70039,3,60,12,1665.0,2200.0,82,1.0,18,NaN,NaN,NaN,NaN,4,75525
36318,GMAD250519,CE00101255,105,21:40:15,CE4E70039,12,60,12,1665.0,2200.0,82,1.0,18,NaN,NaN,NaN,NaN,4,78015
36319,GMAD250519,CE00101255,105,21:40:15,CE4E70039,2,60,12,NaN,NaN,82,1.0,19,NaN,NaN,NaN,NaN,4,78015


IPH

In [6]:
#IPH GM

iph_zonal = pd.read_csv(f'Z:/01 base_datos/12 iph/{dia}_iph_zonal.csv')
iph_troncal = pd.read_csv(f'Z:/01 base_datos/34 iph alim/{dia}_iph_alim.csv')

#Concatenado de dataframes IPH Completa

iph = pd.concat([iph_zonal,iph_troncal], ignore_index=True)

# Rellenar valores faltantes en Sublinea y Ruta basados en TipoDia, ServBus, y Coche
iph[['Sublinea', 'Ruta']] = iph.groupby(['TipoDia', 'ServBus', 'Coche'])[['Sublinea', 'Ruta']].transform(lambda x: x.ffill().bfill())

# Rellenar valores faltantes en ServicioCondEnt y ServicioCondSal basados en TipoDia, ServBus, Linea, Ruta, Coche y Viaje
iph[['ServicioCondEnt', 'ServicioCondSal']] = iph.groupby(['TipoDia', 'ServBus', 'Linea', 'Ruta', 'Coche'])[['ServicioCondEnt', 'ServicioCondSal']].transform(lambda x: x.ffill().bfill())

iph

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_31864\999564515.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  iph[['ServicioCondEnt', 'ServicioCondSal']] = iph.groupby(['TipoDia', 'ServBus', 'Linea', 'Ruta', 'Coche'])[['ServicioCondEnt', 'ServicioCondSal']].transform(lambda x: x.ffill().bfill())


,JornadaTipo,TipoDia,Operador,Instante,ServBus,Evento,Linea,Coche,Sublinea,Ruta,Punto,TipoNodo,Viaje,ServicioCondEnt,TurnoEnt,OperadorEnt,ServicioCondSal,TipoVehiculo
0,GC250624T2,CN00111830,105,3:20:00,CNLYU0001,18,1300,1,4097.0,5658.0,142,5.0,1,CE130330,1.0,105.0,CE130330,8
1,GC250624T2,CN00111830,105,4:00:00,CNLYU0001,4,1300,1,4097.0,5658.0,52372,1.0,1,CE130330,NaN,NaN,CE130330,8
2,GC250624T2,CN00111830,105,4:00:00,CNLYU0001,11,1300,1,4097.0,5658.0,52372,1.0,2,CE130330,NaN,NaN,CE130330,8
3,GC250624T2,CN00111830,105,4:08:48,CNLYU0001,0,1300,1,4097.0,5658.0,52802,1.0,2,CE130330,NaN,NaN,CE130330,8
4,GC250624T2,CN00111830,105,4:17:36,CNLYU0001,0,1300,1,4097.0,5658.0,53222,1.0,2,CE130330,NaN,NaN,CE130330,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36316,GMAD250519,CE00101255,105,20:58:45,CE4E70039,4,60,12,1665.0,2200.0,82,1.0,17,CE101126,NaN,NaN,CE101092,4
36317,GMAD250519,CE00101255,105,20:58:45,CE4E70039,3,60,12,1665.0,2200.0,82,1.0,18,CE101126,NaN,NaN,CE101092,4
36318,GMAD250519,CE00101255,105,21:40:15,CE4E70039,12,60,12,1665.0,2200.0,82,1.0,18,CE101126,NaN,NaN,CE101092,4
36319,GMAD250519,CE00101255,105,21:40:15,CE4E70039,2,60,12,1665.0,2200.0,82,1.0,19,CE101126,NaN,NaN,CE101092,4


In [7]:
# Filtrar el DataFrame para que solo incluya filas donde 'Evento' sea 3 o 11
iph= iph[(iph['Evento'] == 3) | (iph['Evento'] == 11)]

iph['Sublinea'] = iph['Sublinea'].astype(int)
iph['Ruta'] = iph['Ruta'].astype(int)

#Eliminar columnas que no se necesitan para el proceso
col_eliminar = ['Operador', 'TurnoEnt', 'OperadorEnt']

iph= iph.drop(columns=col_eliminar)

#Completar columnas con 0

iph['ServicioCondEnt'].fillna(0, inplace=True)
iph['ServicioCondSal'].fillna(0, inplace=True)

iph

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_31864\602394785.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  iph['Sublinea'] = iph['Sublinea'].astype(int)
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_31864\602394785.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  iph['Ruta'] = iph['Ruta'].astype(int)
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_31864\602394785.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment us

,JornadaTipo,TipoDia,Instante,ServBus,Evento,Linea,Coche,Sublinea,Ruta,Punto,TipoNodo,Viaje,ServicioCondEnt,ServicioCondSal,TipoVehiculo
2,GC250624T2,CN00111830,4:00:00,CNLYU0001,11,1300,1,4097,5658,52372,1.0,2,CE130330,CE130330,8
15,GC250624T2,CN00111830,5:52:00,CNLYU0001,3,1300,1,4097,5658,52372,1.0,3,CE130996,CE130330,8
35,GC250624T2,CN00111830,8:38:00,CNLYU0001,3,1300,1,4097,5658,52372,1.0,4,CE130349,CE130349,8
41,GC250624T2,CN00111830,11:38:00,CNLYU0001,3,1300,1,4097,5658,52372,1.0,5,CE130948,CE130349,8
54,GC250624T2,CN00111830,14:34:00,CNLYU0001,3,1300,1,4097,5658,52372,1.0,6,CE131009,CE130948,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36309,GMAD250519,CE00101255,16:53:00,CE4E70039,3,60,12,1665,2200,82,1.0,14,CE101126,CE101092,4
36311,GMAD250519,CE00101255,18:05:45,CE4E70039,3,60,12,1665,2200,82,1.0,15,CE101126,CE101092,4
36313,GMAD250519,CE00101255,19:15:45,CE4E70039,3,60,12,1665,2200,82,1.0,16,CE101126,CE101092,4
36315,GMAD250519,CE00101255,20:13:00,CE4E70039,3,60,12,1665,2200,82,1.0,17,CE101126,CE101092,4


Matriz de distancia

In [8]:
# Ruta de la carpeta 
matriz_distancia = pd.read_csv(f'Z:/01 base_datos/06 matriz_distancia/{dia}_matriz distancias.csv', encoding='latin')
matriz_distancia_t = pd.read_csv(f'Z:/01 base_datos/31 matriz_distancia_troncal/{dia}_matriz_distancias_troncal.csv', encoding='latin')

#Concatenado de dataframes

md = pd.concat([matriz_distancia, matriz_distancia_t], ignore_index=True)

#Eliminar columnas que no se necesitan para el proceso

col_eliminar = ['Macro', 'ConfigAct', 'ConfigFecha', 'TipoNodo', 'TipoServicio']

md= md.drop(columns=col_eliminar)

#Dividir columna de StrLinea en línea y ruta comercial
md[['linea', 'Ruta_com']] = md['StrLinea'].str.split(' ', n=1, expand=True)
md['linea'] = md['linea'].str.replace('[\[\]]', '', regex=True)

md

<>:17: SyntaxWarning: invalid escape sequence '\['
<>:17: SyntaxWarning: invalid escape sequence '\['
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_31864\3911382657.py:17: SyntaxWarning: invalid escape sequence '\['
  md['linea'] = md['linea'].str.replace('[\[\]]', '', regex=True)


,Linea,StrLinea,Sublinea,Ruta,StrRuta,Nodo,Posicion,Nombre,StrTipoServicio,linea,Ruta_com
0,1029,[1029] 17-1,3611,4497,[4497] 17-1_20201013_V1,50417,0,375B06_Estación Av. Rojas,[0] URBANO,1029,17-1
1,1029,[1029] 17-1,3611,4497,[4497] 17-1_20201013_V1,52982,371,060A06_La Esperanza Norte,[0] URBANO,1029,17-1
2,1029,[1029] 17-1,3611,4497,[4497] 17-1_20201013_V1,52635,1316,165A05_Colegio Militar Simón Bolívar,[0] URBANO,1029,17-1
3,1029,[1029] 17-1,3611,4497,[4497] 17-1_20201013_V1,52434,1826,099A05_Br. Normandía,[0] URBANO,1029,17-1
4,1029,[1029] 17-1,3611,4497,[4497] 17-1_20201013_V1,52433,2026,098A05_Br. Normandía,[0] URBANO,1029,17-1
...,...,...,...,...,...,...,...,...,...,...,...
22919,10650,[10650] M85 - N85,1633,11906,[11906] N85-M85_Ciclovia_V1,61917,14718,Salitre - El Greco A - 2 ó 5,[5] PADRON,10650,M85 - N85
22920,10650,[10650] M85 - N85,1633,11906,[11906] N85-M85_Ciclovia_V1,61858,15183,CAN B - 1 ó 5,[5] PADRON,10650,M85 - N85
22921,10650,[10650] M85 - N85,1633,11906,[11906] N85-M85_Ciclovia_V1,61912,16414,Quinta Paredes B -1 ó 5,[5] PADRON,10650,M85 - N85
22922,10650,[10650] M85 - N85,1633,11906,[11906] N85-M85_Ciclovia_V1,61867,19244,Centro Memoria A - 2 ó 5,[5] PADRON,10650,M85 - N85


In [9]:
# N Parada

def calcular_valor(row):
    filtro = (
        (md['Posicion'] < row['Posicion']) &
        (md['Ruta'] == row['Ruta']) &
        (md['Sublinea'] == row['Sublinea'])
    )
    conteo = len(md.loc[filtro])
    return conteo + 1

# Aplicamos la función a cada fila del DataFrame 'Matriz_de_Distancia'
md['N Parada'] = md.apply(calcular_valor, axis=1)

md['Nodo'] = md['Nodo'].astype(int)

#Matriz_de_Distancia.to_csv(ruta,index=False )

md

,Linea,StrLinea,Sublinea,Ruta,StrRuta,Nodo,Posicion,Nombre,StrTipoServicio,linea,Ruta_com,N Parada
0,1029,[1029] 17-1,3611,4497,[4497] 17-1_20201013_V1,50417,0,375B06_Estación Av. Rojas,[0] URBANO,1029,17-1,1
1,1029,[1029] 17-1,3611,4497,[4497] 17-1_20201013_V1,52982,371,060A06_La Esperanza Norte,[0] URBANO,1029,17-1,2
2,1029,[1029] 17-1,3611,4497,[4497] 17-1_20201013_V1,52635,1316,165A05_Colegio Militar Simón Bolívar,[0] URBANO,1029,17-1,3
3,1029,[1029] 17-1,3611,4497,[4497] 17-1_20201013_V1,52434,1826,099A05_Br. Normandía,[0] URBANO,1029,17-1,4
4,1029,[1029] 17-1,3611,4497,[4497] 17-1_20201013_V1,52433,2026,098A05_Br. Normandía,[0] URBANO,1029,17-1,5
...,...,...,...,...,...,...,...,...,...,...,...,...
22919,10650,[10650] M85 - N85,1633,11906,[11906] N85-M85_Ciclovia_V1,61917,14718,Salitre - El Greco A - 2 ó 5,[5] PADRON,10650,M85 - N85,13
22920,10650,[10650] M85 - N85,1633,11906,[11906] N85-M85_Ciclovia_V1,61858,15183,CAN B - 1 ó 5,[5] PADRON,10650,M85 - N85,14
22921,10650,[10650] M85 - N85,1633,11906,[11906] N85-M85_Ciclovia_V1,61912,16414,Quinta Paredes B -1 ó 5,[5] PADRON,10650,M85 - N85,15
22922,10650,[10650] M85 - N85,1633,11906,[11906] N85-M85_Ciclovia_V1,61867,19244,Centro Memoria A - 2 ó 5,[5] PADRON,10650,M85 - N85,16


Acciones de regulacion

In [10]:
#Importar archivos de acciones reg Zonal - TM

carpeta_origen = 'Z:/01 base_datos/41 Informe Diario CCZ/Blogger/Acciones regulacion'

dataframes = [] # Lista para almacenar cada archivo

fecha_inicio = f'{dia_i}'  
fecha_fin = f'{dia_f}'  

for archivo in os.listdir(carpeta_origen):
    if archivo.endswith('.csv'):  # Filtra solo los archivos con extensión .csv
        ruta_archivo = os.path.join(carpeta_origen, archivo)
        df = pd.read_csv(ruta_archivo, encoding='latin')
        
        # Obtener la fecha del nombre del archivo
        fecha_archivo = pd.to_datetime(archivo.split('_')[0], format='%Y%m%d', errors='coerce')

        # Filtrar por fechas
        if fecha_archivo >= pd.to_datetime(fecha_inicio, format='%d/%m/%Y') and fecha_archivo <= pd.to_datetime(fecha_fin, format='%d/%m/%Y'):
            dataframes.append(df)  # Agregar el DataFrame a la lista

# Combinar todos los DataFrames en uno solo
if dataframes:
    acciones_rev = pd.concat(dataframes)

    # Imprimir el resultado
    acciones_rev
else:
    print("No se encontraron archivos que cumplan con el filtro de fechas.")
    
acciones_rev

No se encontraron archivos que cumplan con el filtro de fechas.


NameError: name 'acciones_rev' is not defined

In [ ]:
# filtrar puesto que empiece por GM o TM
acciones_rev = acciones_rev[acciones_rev['Puesto'].str.startswith(('GM', 'TM'), na=False)]

# Ver los primeros registros filtrados
acciones_rev.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo
0,28/05/2025,2:13:04,0,0,NaN,0,20,Cambiar Conductor,CRSC,Sonia Constanza Correa Rodriguez,GMZ10,"<CambiarConductor ServicioConductor=""CE130053""...",0,Motivo no definido
1,28/05/2025,2:13:15,0,0,NaN,0,20,Cambiar Conductor,CRSC,Sonia Constanza Correa Rodriguez,GMZ10,"<CambiarConductor ServicioConductor=""CE130535""...",0,Motivo no definido
2,28/05/2025,2:13:25,0,0,NaN,0,20,Cambiar Conductor,CRSC,Sonia Constanza Correa Rodriguez,GMZ10,"<CambiarConductor ServicioConductor=""CE130053""...",0,Motivo no definido
3,28/05/2025,2:14:01,0,0,NaN,0,20,Cambiar Conductor,CRSC,Sonia Constanza Correa Rodriguez,GMZ10,"<CambiarConductor ServicioConductor=""CE130054""...",0,Motivo no definido
4,28/05/2025,2:14:12,0,0,NaN,0,20,Cambiar Conductor,JIAG,Jinneth Alexandra Guzman,GMZ04,"<CambiarConductor ServicioConductor=""CE107275""...",0,Motivo no definido


In [ ]:
print(acciones_rev['Puesto'].unique())

['GMZ10' 'GMZ04' 'GMZ09' 'GMZ08' 'GMZ13' 'GMT01' 'GMZ06' 'GMZ03' 'GMZ01'
 'GMZ05' 'GMZ02']


In [ ]:
acciones_rev3  = acciones_rev.copy()

acciones_rev3

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo
0,28/05/2025,2:13:04,0,0,NaN,0,20,Cambiar Conductor,CRSC,Sonia Constanza Correa Rodriguez,GMZ10,"<CambiarConductor ServicioConductor=""CE130053""...",0,Motivo no definido
1,28/05/2025,2:13:15,0,0,NaN,0,20,Cambiar Conductor,CRSC,Sonia Constanza Correa Rodriguez,GMZ10,"<CambiarConductor ServicioConductor=""CE130535""...",0,Motivo no definido
2,28/05/2025,2:13:25,0,0,NaN,0,20,Cambiar Conductor,CRSC,Sonia Constanza Correa Rodriguez,GMZ10,"<CambiarConductor ServicioConductor=""CE130053""...",0,Motivo no definido
3,28/05/2025,2:14:01,0,0,NaN,0,20,Cambiar Conductor,CRSC,Sonia Constanza Correa Rodriguez,GMZ10,"<CambiarConductor ServicioConductor=""CE130054""...",0,Motivo no definido
4,28/05/2025,2:14:12,0,0,NaN,0,20,Cambiar Conductor,JIAG,Jinneth Alexandra Guzman,GMZ04,"<CambiarConductor ServicioConductor=""CE107275""...",0,Motivo no definido
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1142,28/05/2025,14:10:24,1557,0,NaN,0,22,Desvio,TAP,Adriana Patricia Tellez,GMZ06,"<ActivarDesvio Desvio=""17265"" Activo=""SI"" Reen...",0,Motivo no definido
1143,28/05/2025,14:10:26,1557,0,NaN,0,22,Desvio,TAP,Adriana Patricia Tellez,GMZ06,"<ActivarDesvio Desvio=""17266"" Activo=""SI"" Reen...",0,Motivo no definido
1144,28/05/2025,14:12:51,1557,25,Z50-4146,504146,36,Cambiar Coche,TAP,Adriana Patricia Tellez,GMZ06,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje
1145,28/05/2025,14:14:43,1091,28,Z50-4140,504140,1,Enviar Mensaje,TAP,Adriana Patricia Tellez,GMZ06,"<EnviarMensaje Mensaje=""DICKSON va en|convoy ...",12,Solicitud del área de programación


In [ ]:
# Filtrar las filas donde 'Accion' sea igual a 5
acciones_rev3 = acciones_rev3[acciones_rev3['Accion'].isin([5])]

acciones_rev3

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo
154,28/05/2025,3:14:01,1182,3,NaN,0,5,Eliminar Coche,JIAG,Jinneth Alexandra Guzman,GMZ04,"<EliminarCoche Servicio=""CEIJW0006"" Motivo=""14...",14,No se presenta operador a realizar servicio
164,28/05/2025,3:50:00,1083,6,NaN,0,5,Eliminar Coche,CRSC,Sonia Constanza Correa Rodriguez,GMZ10,"<EliminarCoche Servicio=""CEIKB0006"" Motivo=""14...",14,No se presenta operador a realizar servicio
170,28/05/2025,4:14:14,1137,8,NaN,0,5,Eliminar Coche,dyrojas1,Deisi Yanira Rojas Torres,GMZ04,"<EliminarCoche Servicio=""CEIH40008"" Motivo=""14...",14,No se presenta operador a realizar servicio
173,28/05/2025,4:27:28,1182,9,NaN,0,5,Eliminar Coche,dyrojas1,Deisi Yanira Rojas Torres,GMZ04,"<EliminarCoche Servicio=""CEIJW0018"" Motivo=""14...",14,No se presenta operador a realizar servicio
176,28/05/2025,4:34:30,1030,15,NaN,0,5,Eliminar Coche,LMVG,Laura Marcela Vargas Gutierrez,GMZ09,"<EliminarCoche Servicio=""CEICV0015"" Motivo=""14...",14,No se presenta operador a realizar servicio
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1090,28/05/2025,13:42:35,1435,2,NaN,0,5,Eliminar Coche,MAPB,Maicol Alexander Pita Blanco,GMT01,"<EliminarCoche Servicio=""CEIG40002"" Motivo=""14...",14,No se presenta operador a realizar servicio
1102,28/05/2025,13:49:30,1030,13,NaN,0,5,Eliminar Coche,MIM,Martha Ibeth Martin,GMZ02,"<EliminarCoche Servicio=""CEICV0013"" Motivo=""14...",14,No se presenta operador a realizar servicio
1108,28/05/2025,13:50:59,1030,13,NaN,0,5,Eliminar Coche,MIM,Martha Ibeth Martin,GMZ02,"<EliminarCoche Servicio=""CEICV0013"" Motivo=""14...",14,No se presenta operador a realizar servicio
1109,28/05/2025,13:51:08,1215,6,NaN,0,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<EliminarCoche Servicio=""CEIBS0005"" Motivo=""1""...",1,Concesionario no envía móvil


In [ ]:
# Llevar ruta_sae a eliminaciones Motivo 31

def calcular_lin(linea):
    
    filtro = (
        (rutas['linea'] == linea) 
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not rutas.loc[filtro].empty:
        # Obtener el primer valor
        tipo = rutas.loc[filtro, 'ruta_sae'].iloc[0]
        return tipo if not pd.isna(tipo) else None 
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
acciones_rev3['ruta_sae'] = acciones_rev3.apply(
    lambda row: calcular_lin(
        row['Linea']
    ),
    axis=1
)
acciones_rev3

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9408\4137193078.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_rev3['ruta_sae'] = acciones_rev3.apply(


,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae
154,28/05/2025,3:14:01,1182,3,NaN,0,5,Eliminar Coche,JIAG,Jinneth Alexandra Guzman,GMZ04,"<EliminarCoche Servicio=""CEIJW0006"" Motivo=""14...",14,No se presenta operador a realizar servicio,8411
164,28/05/2025,3:50:00,1083,6,NaN,0,5,Eliminar Coche,CRSC,Sonia Constanza Correa Rodriguez,GMZ10,"<EliminarCoche Servicio=""CEIKB0006"" Motivo=""14...",14,No se presenta operador a realizar servicio,4788
170,28/05/2025,4:14:14,1137,8,NaN,0,5,Eliminar Coche,dyrojas1,Deisi Yanira Rojas Torres,GMZ04,"<EliminarCoche Servicio=""CEIH40008"" Motivo=""14...",14,No se presenta operador a realizar servicio,5024
173,28/05/2025,4:27:28,1182,9,NaN,0,5,Eliminar Coche,dyrojas1,Deisi Yanira Rojas Torres,GMZ04,"<EliminarCoche Servicio=""CEIJW0018"" Motivo=""14...",14,No se presenta operador a realizar servicio,8411
176,28/05/2025,4:34:30,1030,15,NaN,0,5,Eliminar Coche,LMVG,Laura Marcela Vargas Gutierrez,GMZ09,"<EliminarCoche Servicio=""CEICV0015"" Motivo=""14...",14,No se presenta operador a realizar servicio,4509
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1090,28/05/2025,13:42:35,1435,2,NaN,0,5,Eliminar Coche,MAPB,Maicol Alexander Pita Blanco,GMT01,"<EliminarCoche Servicio=""CEIG40002"" Motivo=""14...",14,No se presenta operador a realizar servicio,6223
1102,28/05/2025,13:49:30,1030,13,NaN,0,5,Eliminar Coche,MIM,Martha Ibeth Martin,GMZ02,"<EliminarCoche Servicio=""CEICV0013"" Motivo=""14...",14,No se presenta operador a realizar servicio,4509
1108,28/05/2025,13:50:59,1030,13,NaN,0,5,Eliminar Coche,MIM,Martha Ibeth Martin,GMZ02,"<EliminarCoche Servicio=""CEICV0013"" Motivo=""14...",14,No se presenta operador a realizar servicio,4509
1109,28/05/2025,13:51:08,1215,6,NaN,0,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<EliminarCoche Servicio=""CEIBS0005"" Motivo=""1""...",1,Concesionario no envía móvil,8103


In [ ]:
# Rellenar los valores vacíos (NaN) en la columna 'ruta_sae' con 1
acciones_rev3['ruta_sae'] = acciones_rev3['ruta_sae'].fillna(1)

acciones_rev3['ruta_sae'] = acciones_rev3['ruta_sae'].astype(int)

# Eliminar las filas donde 'ruta_sae' sea igual a 1
acciones_rev3 = acciones_rev3[acciones_rev3['ruta_sae'] != 1]

# Verificar los cambios
acciones_rev3.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9408\884166993.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_rev3['ruta_sae'] = acciones_rev3['ruta_sae'].fillna(1)
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9408\884166993.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_rev3['ruta_sae'] = acciones_rev3['ruta_sae'].astype(int)


,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae
154,28/05/2025,3:14:01,1182,3,NaN,0,5,Eliminar Coche,JIAG,Jinneth Alexandra Guzman,GMZ04,"<EliminarCoche Servicio=""CEIJW0006"" Motivo=""14...",14,No se presenta operador a realizar servicio,8411
164,28/05/2025,3:50:00,1083,6,NaN,0,5,Eliminar Coche,CRSC,Sonia Constanza Correa Rodriguez,GMZ10,"<EliminarCoche Servicio=""CEIKB0006"" Motivo=""14...",14,No se presenta operador a realizar servicio,4788
170,28/05/2025,4:14:14,1137,8,NaN,0,5,Eliminar Coche,dyrojas1,Deisi Yanira Rojas Torres,GMZ04,"<EliminarCoche Servicio=""CEIH40008"" Motivo=""14...",14,No se presenta operador a realizar servicio,5024
173,28/05/2025,4:27:28,1182,9,NaN,0,5,Eliminar Coche,dyrojas1,Deisi Yanira Rojas Torres,GMZ04,"<EliminarCoche Servicio=""CEIJW0018"" Motivo=""14...",14,No se presenta operador a realizar servicio,8411
176,28/05/2025,4:34:30,1030,15,NaN,0,5,Eliminar Coche,LMVG,Laura Marcela Vargas Gutierrez,GMZ09,"<EliminarCoche Servicio=""CEICV0015"" Motivo=""14...",14,No se presenta operador a realizar servicio,4509


In [ ]:
#concatenar los dataframe
acciones = acciones_rev3.copy()

acciones.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae
154,28/05/2025,3:14:01,1182,3,NaN,0,5,Eliminar Coche,JIAG,Jinneth Alexandra Guzman,GMZ04,"<EliminarCoche Servicio=""CEIJW0006"" Motivo=""14...",14,No se presenta operador a realizar servicio,8411
164,28/05/2025,3:50:00,1083,6,NaN,0,5,Eliminar Coche,CRSC,Sonia Constanza Correa Rodriguez,GMZ10,"<EliminarCoche Servicio=""CEIKB0006"" Motivo=""14...",14,No se presenta operador a realizar servicio,4788
170,28/05/2025,4:14:14,1137,8,NaN,0,5,Eliminar Coche,dyrojas1,Deisi Yanira Rojas Torres,GMZ04,"<EliminarCoche Servicio=""CEIH40008"" Motivo=""14...",14,No se presenta operador a realizar servicio,5024
173,28/05/2025,4:27:28,1182,9,NaN,0,5,Eliminar Coche,dyrojas1,Deisi Yanira Rojas Torres,GMZ04,"<EliminarCoche Servicio=""CEIJW0018"" Motivo=""14...",14,No se presenta operador a realizar servicio,8411
176,28/05/2025,4:34:30,1030,15,NaN,0,5,Eliminar Coche,LMVG,Laura Marcela Vargas Gutierrez,GMZ09,"<EliminarCoche Servicio=""CEICV0015"" Motivo=""14...",14,No se presenta operador a realizar servicio,4509


In [ ]:
# Ordenar el DataFrame por las columnas Fecha, Instante, Linea y Coche
acciones = acciones.sort_values(by=['Fecha', 'Instante', 'Linea', 'Coche'], ascending=True)

acciones.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae
658,28/05/2025,10:01:34,1413,8,NaN,0,5,Eliminar Coche,TFW,Wilson Torres Fajardo,GMZ01,"<EliminarCoche Servicio=""CEIKC0008"" Motivo=""14...",14,No se presenta operador a realizar servicio,8390
669,28/05/2025,10:03:33,1557,3,NaN,0,5,Eliminar Coche,ECM,Esteban Camilo Medina,GMZ06,"<EliminarCoche Servicio=""CEIHO0003"" Motivo=""2""...",2,Bus varado en la vía,7911
700,28/05/2025,10:27:46,1210,12,NaN,0,5,Eliminar Coche,PAE,Edith Patarroyo,GMZ05,"<EliminarCoche Servicio=""CEIB70008"" Motivo=""14...",14,No se presenta operador a realizar servicio,6873
703,28/05/2025,10:31:12,1185,22,NaN,0,5,Eliminar Coche,VSC,Sandra Carolina Valeriano,GMZ08,"<EliminarCoche Servicio=""CEIHM0023"" Motivo=""14...",14,No se presenta operador a realizar servicio,8348
704,28/05/2025,10:31:27,1557,19,NaN,0,5,Eliminar Coche,ECM,Esteban Camilo Medina,GMZ06,"<EliminarCoche Servicio=""CEIHO0019"" Motivo=""14...",14,No se presenta operador a realizar servicio,7911


In [ ]:
# Filtrar filas donde los valores de la columna 'NumBus' comiencen con 'Z50'
acciones = acciones[acciones['CodigoBus'].astype(str).str.startswith('Z50', 'Z52')]

# Reiniciar el índice después del filtrado
acciones.reset_index(drop=True, inplace=True)

acciones.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae
0,28/05/2025,10:41:08,1091,40,Z50-4411,504411,5,Eliminar Coche,ECM,Esteban Camilo Medina,GMZ06,"<EliminarCoche Servicio=""CEIK5G019"" Motivo=""19...",19,Fallas del SIRCI,8442
1,28/05/2025,11:33:43,1091,27,Z50-4347,504347,5,Eliminar Coche,ECM,Esteban Camilo Medina,GMZ06,"<EliminarCoche Servicio=""CEIK50027"" Motivo=""2""...",2,Bus varado en la vía,8442
2,28/05/2025,12:39:57,1182,29,Z50-4485,504485,5,Eliminar Coche,GWM,William Munar Gonzalez,GMZ04,"<EliminarCoche Servicio=""CEIJW0029"" Motivo=""14...",14,No se presenta operador a realizar servicio,8411
3,28/05/2025,13:01:24,1083,1,Z50-7077,507077,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<EliminarCoche Servicio=""CEIKB0001"" Motivo=""2""...",2,Bus varado en la vía,4788
4,28/05/2025,13:32:34,1215,6,Z50-4259,504259,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<EliminarCoche Servicio=""CEIBS0005"" Motivo=""10...",10,Accidente,8103


In [ ]:
# Convertir la columna 'Fecha' a formato datetime
acciones['Fecha'] = pd.to_datetime(acciones['Fecha'], errors='coerce')

acciones.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9408\692094097.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  acciones['Fecha'] = pd.to_datetime(acciones['Fecha'], errors='coerce')


,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae
0,2025-05-28,10:41:08,1091,40,Z50-4411,504411,5,Eliminar Coche,ECM,Esteban Camilo Medina,GMZ06,"<EliminarCoche Servicio=""CEIK5G019"" Motivo=""19...",19,Fallas del SIRCI,8442
1,2025-05-28,11:33:43,1091,27,Z50-4347,504347,5,Eliminar Coche,ECM,Esteban Camilo Medina,GMZ06,"<EliminarCoche Servicio=""CEIK50027"" Motivo=""2""...",2,Bus varado en la vía,8442
2,2025-05-28,12:39:57,1182,29,Z50-4485,504485,5,Eliminar Coche,GWM,William Munar Gonzalez,GMZ04,"<EliminarCoche Servicio=""CEIJW0029"" Motivo=""14...",14,No se presenta operador a realizar servicio,8411
3,2025-05-28,13:01:24,1083,1,Z50-7077,507077,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<EliminarCoche Servicio=""CEIKB0001"" Motivo=""2""...",2,Bus varado en la vía,4788
4,2025-05-28,13:32:34,1215,6,Z50-4259,504259,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<EliminarCoche Servicio=""CEIBS0005"" Motivo=""10...",10,Accidente,8103


In [ ]:
# Definir una función para contar los caracteres en la cadena
def contar_caracteres(cadena):
    return len(cadena)

# Aplicar la función a la columna "Parametros" solo cuando la columna "Accion" es igual a 5
acciones['Conteo_Caracteres'] = acciones.loc[acciones['Accion'] == 5, 'Parametros'].apply(contar_caracteres)

# 1. Extraer cadena de servicio
def extraer_servicio_accion5(cadena, accion):
    if accion != 5:
        return None
    
    # Patrón para buscar el servicio
    patron_servicio = r'Servicio="([^"]*)"'
    
    # Buscar el servicio en la cadena
    servicio_match = re.search(patron_servicio, cadena)
    
    # Extraer el servicio si se encuentra
    servicio = servicio_match.group(1) if servicio_match else None
    
    return servicio

# Aplicar la función a la columna "Parametros"
acciones['Servicio_eliminado'] = acciones.apply(lambda row: extraer_servicio_accion5(row['Parametros'], row['Accion']), axis=1)

acciones.head(5)

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae,Conteo_Caracteres,Servicio_eliminado
0,2025-05-28,10:41:08,1091,40,Z50-4411,504411,5,Eliminar Coche,ECM,Esteban Camilo Medina,GMZ06,"<EliminarCoche Servicio=""CEIK5G019"" Motivo=""19...",19,Fallas del SIRCI,8442,505,CEIK5G019
1,2025-05-28,11:33:43,1091,27,Z50-4347,504347,5,Eliminar Coche,ECM,Esteban Camilo Medina,GMZ06,"<EliminarCoche Servicio=""CEIK50027"" Motivo=""2""...",2,Bus varado en la vía,8442,504,CEIK50027
2,2025-05-28,12:39:57,1182,29,Z50-4485,504485,5,Eliminar Coche,GWM,William Munar Gonzalez,GMZ04,"<EliminarCoche Servicio=""CEIJW0029"" Motivo=""14...",14,No se presenta operador a realizar servicio,8411,501,CEIJW0029
3,2025-05-28,13:01:24,1083,1,Z50-7077,507077,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<EliminarCoche Servicio=""CEIKB0001"" Motivo=""2""...",2,Bus varado en la vía,4788,503,CEIKB0001
4,2025-05-28,13:32:34,1215,6,Z50-4259,504259,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<EliminarCoche Servicio=""CEIBS0005"" Motivo=""10...",10,Accidente,8103,501,CEIBS0005


In [ ]:
#Verificar el motivo de eliminación del servicio
def determinar_valor(cadena):
    conteo = len(cadena)
    if conteo <= 125:
        return "EliminarCoche"
    elif 125 < conteo <= 320:
        return "Desde Ruta / PosicionParada/Retirada Cochera"
    elif 320 < conteo <= 355:
        return "Desde Ruta / Hasta Ruta"
    elif 355 < conteo <= 365:
        return "Desde Ruta / PosicionParada/Hasta Ruta"
    elif 365 < conteo <= 430:
        return "Desde Ruta / Incorporacion Cochera/Hasta Ruta"
    elif 430 < conteo <= 455:
        return "Desde Ruta / PosicionParada/Retirada Cochera/Hasta Ruta"
    elif 455 < conteo <= 505:
        return "Desde Ruta / Retirada Cochera/ Incorporacion Cochera/Hasta Ruta"
    else:
        return "Desde Ruta / Posicion Parada / Retirada Cochera / Incorporacion Cochera/Hasta Ruta"

# Aplicar la función a la columna "Parametros" solo cuando la columna "Accion" es igual a 5
acciones['Forma_eliminada'] = acciones.loc[acciones['Accion'] == 5, 'Parametros'].apply(determinar_valor)

acciones.head(5)

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae,Conteo_Caracteres,Servicio_eliminado,Forma_eliminada
0,2025-05-28,10:41:08,1091,40,Z50-4411,504411,5,Eliminar Coche,ECM,Esteban Camilo Medina,GMZ06,"<EliminarCoche Servicio=""CEIK5G019"" Motivo=""19...",19,Fallas del SIRCI,8442,505,CEIK5G019,Desde Ruta / Retirada Cochera/ Incorporacion C...
1,2025-05-28,11:33:43,1091,27,Z50-4347,504347,5,Eliminar Coche,ECM,Esteban Camilo Medina,GMZ06,"<EliminarCoche Servicio=""CEIK50027"" Motivo=""2""...",2,Bus varado en la vía,8442,504,CEIK50027,Desde Ruta / Retirada Cochera/ Incorporacion C...
2,2025-05-28,12:39:57,1182,29,Z50-4485,504485,5,Eliminar Coche,GWM,William Munar Gonzalez,GMZ04,"<EliminarCoche Servicio=""CEIJW0029"" Motivo=""14...",14,No se presenta operador a realizar servicio,8411,501,CEIJW0029,Desde Ruta / Retirada Cochera/ Incorporacion C...
3,2025-05-28,13:01:24,1083,1,Z50-7077,507077,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<EliminarCoche Servicio=""CEIKB0001"" Motivo=""2""...",2,Bus varado en la vía,4788,503,CEIKB0001,Desde Ruta / Retirada Cochera/ Incorporacion C...
4,2025-05-28,13:32:34,1215,6,Z50-4259,504259,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<EliminarCoche Servicio=""CEIBS0005"" Motivo=""10...",10,Accidente,8103,501,CEIBS0005,Desde Ruta / Retirada Cochera/ Incorporacion C...


In [ ]:
# Obtener el conteo de cada valor único en la columna 'Servicio_eliminado'
conteo_valores = acciones['Servicio_eliminado'].value_counts()

# Mapear los valores de la columna 'Servicio_eliminado' al conteo correspondiente
acciones['Conteo_Servicio_eliminado'] = acciones['Servicio_eliminado'].map(conteo_valores)

# Mostrar el DataFrame con la nueva columna 'Conteo_Servicio_eliminado'
acciones.head(5)

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae,Conteo_Caracteres,Servicio_eliminado,Forma_eliminada,Conteo_Servicio_eliminado
0,2025-05-28,10:41:08,1091,40,Z50-4411,504411,5,Eliminar Coche,ECM,Esteban Camilo Medina,GMZ06,"<EliminarCoche Servicio=""CEIK5G019"" Motivo=""19...",19,Fallas del SIRCI,8442,505,CEIK5G019,Desde Ruta / Retirada Cochera/ Incorporacion C...,1
1,2025-05-28,11:33:43,1091,27,Z50-4347,504347,5,Eliminar Coche,ECM,Esteban Camilo Medina,GMZ06,"<EliminarCoche Servicio=""CEIK50027"" Motivo=""2""...",2,Bus varado en la vía,8442,504,CEIK50027,Desde Ruta / Retirada Cochera/ Incorporacion C...,1
2,2025-05-28,12:39:57,1182,29,Z50-4485,504485,5,Eliminar Coche,GWM,William Munar Gonzalez,GMZ04,"<EliminarCoche Servicio=""CEIJW0029"" Motivo=""14...",14,No se presenta operador a realizar servicio,8411,501,CEIJW0029,Desde Ruta / Retirada Cochera/ Incorporacion C...,1
3,2025-05-28,13:01:24,1083,1,Z50-7077,507077,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<EliminarCoche Servicio=""CEIKB0001"" Motivo=""2""...",2,Bus varado en la vía,4788,503,CEIKB0001,Desde Ruta / Retirada Cochera/ Incorporacion C...,1
4,2025-05-28,13:32:34,1215,6,Z50-4259,504259,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<EliminarCoche Servicio=""CEIBS0005"" Motivo=""10...",10,Accidente,8103,501,CEIBS0005,Desde Ruta / Retirada Cochera/ Incorporacion C...,1


In [ ]:
# Encontrar la posición de "Viaje=" en la columna 'Parámetros'
acciones['Posicion'] = acciones['Parametros'].str.find("IdViaje=")

# Extraer los últimos 2 caracteres de la subcadena a partir de la posición encontrada
acciones['Viaje Ini'] = acciones.apply(lambda row: row['Parametros'][row['Posicion']+8:row['Posicion']+11], axis=1)

# Reemplazar las comillas simples (') por un valor en blanco
acciones['Viaje Ini'] = acciones['Viaje Ini'].str.replace('"', '')

# Eliminar espacios en blanco al principio o al final de la cadena
acciones['Viaje Ini'] = acciones['Viaje Ini'].str.strip()

acciones.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Parametros,Motivo,DescripcionMotivo,ruta_sae,Conteo_Caracteres,Servicio_eliminado,Forma_eliminada,Conteo_Servicio_eliminado,Posicion,Viaje Ini
0,2025-05-28,10:41:08,1091,40,Z50-4411,504411,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,"<EliminarCoche Servicio=""CEIK5G019"" Motivo=""19...",19,Fallas del SIRCI,8442,505,CEIK5G019,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,114,1
1,2025-05-28,11:33:43,1091,27,Z50-4347,504347,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,"<EliminarCoche Servicio=""CEIK50027"" Motivo=""2""...",2,Bus varado en la vía,8442,504,CEIK50027,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,113,3
2,2025-05-28,12:39:57,1182,29,Z50-4485,504485,5,Eliminar Coche,GWM,William Munar Gonzalez,...,"<EliminarCoche Servicio=""CEIJW0029"" Motivo=""14...",14,No se presenta operador a realizar servicio,8411,501,CEIJW0029,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,114,6
3,2025-05-28,13:01:24,1083,1,Z50-7077,507077,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,"<EliminarCoche Servicio=""CEIKB0001"" Motivo=""2""...",2,Bus varado en la vía,4788,503,CEIKB0001,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,112,4
4,2025-05-28,13:32:34,1215,6,Z50-4259,504259,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,"<EliminarCoche Servicio=""CEIBS0005"" Motivo=""10...",10,Accidente,8103,501,CEIBS0005,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,113,6


In [ ]:
# Encontrar la posición de "Viaje=" en la columna 'Parámetros'
acciones['Posicion1'] = acciones['Parametros'].str.find("ViajeLinea=")

# Extraer los últimos 2 caracteres de la subcadena a partir de la posición encontrada
acciones['Viaje Ini1'] = acciones.apply(lambda row: row['Parametros'][row['Posicion1']+11:row['Posicion1']+14], axis=1)

# Reemplazar las comillas simples (') por un valor en blanco
acciones['Viaje Ini1'] = acciones['Viaje Ini1'].str.replace('"', '')

# Eliminar espacios en blanco al principio o al final de la cadena
acciones['Viaje Ini1'] = acciones['Viaje Ini1'].str.strip()

acciones.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,DescripcionMotivo,ruta_sae,Conteo_Caracteres,Servicio_eliminado,Forma_eliminada,Conteo_Servicio_eliminado,Posicion,Viaje Ini,Posicion1,Viaje Ini1
0,2025-05-28,10:41:08,1091,40,Z50-4411,504411,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,Fallas del SIRCI,8442,505,CEIK5G019,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,114,1,126,1
1,2025-05-28,11:33:43,1091,27,Z50-4347,504347,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,Bus varado en la vía,8442,504,CEIK50027,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,113,3,125,2
2,2025-05-28,12:39:57,1182,29,Z50-4485,504485,5,Eliminar Coche,GWM,William Munar Gonzalez,...,No se presenta operador a realizar servicio,8411,501,CEIJW0029,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,114,6,126,5
3,2025-05-28,13:01:24,1083,1,Z50-7077,507077,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,Bus varado en la vía,4788,503,CEIKB0001,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,112,4,124,3
4,2025-05-28,13:32:34,1215,6,Z50-4259,504259,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,Accidente,8103,501,CEIBS0005,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,113,6,125,9


In [ ]:
# Función para extraer 2 caracteres después de la segunda aparición de "Viaje="
def extract_viaje(row):
    matches = re.finditer(r'IdViaje=', row['Parametros'])
    # Encontrar la segunda coincidencia
    match = next(matches, None)
    if match:
        match = next(matches, None)  # Segunda coincidencia
        if match:
            start = match.end()  # Posición de inicio después de la segunda coincidencia
            return row['Parametros'][start:start+4]  # Extraer 4 caracteres
    return None

# Aplicar la función a la columna Parametros y crear una nueva columna 'Viaje_extraido'
acciones['Viaje_fin'] = acciones.apply(extract_viaje, axis=1)

# Reemplazar las comillas simples (') por un valor en blanco
acciones['Viaje_fin'] = acciones['Viaje_fin'].str.replace('"', '')

# Eliminar espacios en blanco al principio o al final de la cadena
acciones['Viaje_fin'] = acciones['Viaje_fin'].str.strip()

acciones.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,ruta_sae,Conteo_Caracteres,Servicio_eliminado,Forma_eliminada,Conteo_Servicio_eliminado,Posicion,Viaje Ini,Posicion1,Viaje Ini1,Viaje_fin
0,2025-05-28,10:41:08,1091,40,Z50-4411,504411,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,8442,505,CEIK5G019,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,114,1,126,1,2
1,2025-05-28,11:33:43,1091,27,Z50-4347,504347,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,8442,504,CEIK50027,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,113,3,125,2,4
2,2025-05-28,12:39:57,1182,29,Z50-4485,504485,5,Eliminar Coche,GWM,William Munar Gonzalez,...,8411,501,CEIJW0029,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,114,6,126,5,8
3,2025-05-28,13:01:24,1083,1,Z50-7077,507077,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,4788,503,CEIKB0001,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,112,4,124,3,5
4,2025-05-28,13:32:34,1215,6,Z50-4259,504259,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,8103,501,CEIBS0005,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,113,6,125,9,7


In [ ]:
# Función para extraer 2 caracteres después de la segunda aparición de "Viaje="
def extract_viaje(row):
    matches = re.finditer(r'ViajeLinea=', row['Parametros'])
    # Encontrar la segunda coincidencia
    match = next(matches, None)
    if match:
        match = next(matches, None)  # Segunda coincidencia
        if match:
            start = match.end()  # Posición de inicio después de la segunda coincidencia
            return row['Parametros'][start:start+4]  # Extraer 4 caracteres
    return None

# Aplicar la función a la columna Parametros y crear una nueva columna 'Viaje_extraido'
acciones['Viaje_fin1'] = acciones.apply(extract_viaje, axis=1)

# Reemplazar las comillas simples (') por un valor en blanco
acciones['Viaje_fin1'] = acciones['Viaje_fin1'].str.replace('"', '')

# Eliminar espacios en blanco al principio o al final de la cadena
acciones['Viaje_fin1'] = acciones['Viaje_fin1'].str.strip()

acciones.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Conteo_Caracteres,Servicio_eliminado,Forma_eliminada,Conteo_Servicio_eliminado,Posicion,Viaje Ini,Posicion1,Viaje Ini1,Viaje_fin,Viaje_fin1
0,2025-05-28,10:41:08,1091,40,Z50-4411,504411,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,505,CEIK5G019,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,114,1,126,1,2,2
1,2025-05-28,11:33:43,1091,27,Z50-4347,504347,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,504,CEIK50027,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,113,3,125,2,4,3
2,2025-05-28,12:39:57,1182,29,Z50-4485,504485,5,Eliminar Coche,GWM,William Munar Gonzalez,...,501,CEIJW0029,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,114,6,126,5,8,7
3,2025-05-28,13:01:24,1083,1,Z50-7077,507077,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,503,CEIKB0001,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,112,4,124,3,5,4
4,2025-05-28,13:32:34,1215,6,Z50-4259,504259,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,501,CEIBS0005,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,113,6,125,9,7,11


In [ ]:
# Encontrar la posición de "Viaje=" en la columna 'Parámetros'
acciones['Posicion'] = acciones['Parametros'].str.find("HoraTeor=")

# Extraer los últimos 2 caracteres de la subcadena a partir de la posición encontrada
acciones['HoraTeorIni'] = acciones.apply(lambda row: row['Parametros'][row['Posicion']+9:row['Posicion']+19], axis=1)

# Reemplazar las comillas simples (') por un valor en blanco
acciones['HoraTeorIni'] = acciones['HoraTeorIni'].str.replace('"', '')

# Eliminar espacios en blanco al principio o al final de la cadena
acciones['HoraTeorIni'] = acciones['HoraTeorIni'].str.strip()

acciones.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Servicio_eliminado,Forma_eliminada,Conteo_Servicio_eliminado,Posicion,Viaje Ini,Posicion1,Viaje Ini1,Viaje_fin,Viaje_fin1,HoraTeorIni
0,2025-05-28,10:41:08,1091,40,Z50-4411,504411,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,CEIK5G019,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,200,1,126,1,2,2,10:34:18
1,2025-05-28,11:33:43,1091,27,Z50-4347,504347,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,CEIK50027,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,199,3,125,2,4,3,11:08:58
2,2025-05-28,12:39:57,1182,29,Z50-4485,504485,5,Eliminar Coche,GWM,William Munar Gonzalez,...,CEIJW0029,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,196,6,126,5,8,7,12:43:00
3,2025-05-28,13:01:24,1083,1,Z50-7077,507077,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,CEIKB0001,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,198,4,124,3,5,4,12:50:10
4,2025-05-28,13:32:34,1215,6,Z50-4259,504259,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,CEIBS0005,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,195,6,125,9,7,11,13:00:00


In [ ]:
# Función para extraer el valor después de la tercera aparición de "HoraFinTeor="
def extract_viaje(row):
    matches = list(re.finditer(r'HoraTeor=', row['Parametros']))
    
    if len(matches) >= 2:  # Verificar que haya al menos 3 coincidencias
        start = matches[1].end()  # Posición después de la tercera coincidencia
        return row['Parametros'][start:start+9]  # Extraer 9 caracteres
    
    return None

# Aplicar la función a la columna 'Parametros' y crear la nueva columna 'HoraFinTeor'
acciones['HoraFinTeor2'] = acciones.apply(extract_viaje, axis=1)

# Limpiar la columna eliminando comillas dobles y espacios en blanco
acciones['HoraFinTeor2'] = acciones['HoraFinTeor2'].str.replace('"', '').str.strip()

acciones.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Forma_eliminada,Conteo_Servicio_eliminado,Posicion,Viaje Ini,Posicion1,Viaje Ini1,Viaje_fin,Viaje_fin1,HoraTeorIni,HoraFinTeor2
0,2025-05-28,10:41:08,1091,40,Z50-4411,504411,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,200,1,126,1,2,2,10:34:18,10:39:18
1,2025-05-28,11:33:43,1091,27,Z50-4347,504347,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,199,3,125,2,4,3,11:08:58,11:13:58
2,2025-05-28,12:39:57,1182,29,Z50-4485,504485,5,Eliminar Coche,GWM,William Munar Gonzalez,...,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,196,6,126,5,8,7,12:43:00,12:44:00
3,2025-05-28,13:01:24,1083,1,Z50-7077,507077,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,198,4,124,3,5,4,12:50:10,12:55:10
4,2025-05-28,13:32:34,1215,6,Z50-4259,504259,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,Desde Ruta / Retirada Cochera/ Incorporacion C...,1,195,6,125,9,7,11,13:00:00,13:05:00


In [ ]:
# Función para extraer el valor después de la tercera aparición de "HoraFinTeor="
def extract_viaje(row):
    matches = list(re.finditer(r'HoraTeor=', row['Parametros']))
    
    if len(matches) >= 3:  # Verificar que haya al menos 3 coincidencias
        start = matches[2].end()  # Posición después de la tercera coincidencia
        return row['Parametros'][start:start+9]  # Extraer 9 caracteres
    
    return None

# Aplicar la función a la columna 'Parametros' y crear la nueva columna 'HoraFinTeor'
acciones['HoraFinTeor3'] = acciones.apply(extract_viaje, axis=1)

# Limpiar la columna eliminando comillas dobles y espacios en blanco
acciones['HoraFinTeor3'] = acciones['HoraFinTeor3'].str.replace('"', '').str.strip()

acciones.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Conteo_Servicio_eliminado,Posicion,Viaje Ini,Posicion1,Viaje Ini1,Viaje_fin,Viaje_fin1,HoraTeorIni,HoraFinTeor2,HoraFinTeor3
0,2025-05-28,10:41:08,1091,40,Z50-4411,504411,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,1,200,1,126,1,2,2,10:34:18,10:39:18,12:01:00
1,2025-05-28,11:33:43,1091,27,Z50-4347,504347,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,1,199,3,125,2,4,3,11:08:58,11:13:58,13:13:30
2,2025-05-28,12:39:57,1182,29,Z50-4485,504485,5,Eliminar Coche,GWM,William Munar Gonzalez,...,1,196,6,126,5,8,7,12:43:00,12:44:00,16:56:30
3,2025-05-28,13:01:24,1083,1,Z50-7077,507077,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,1,198,4,124,3,5,4,12:50:10,12:55:10,17:20:30
4,2025-05-28,13:32:34,1215,6,Z50-4259,504259,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,1,195,6,125,9,7,11,13:00:00,13:05:00,15:00:15


In [ ]:
# Función para extraer el valor después de la tercera aparición de "HoraFinTeor="
def extract_viaje(row):
    matches = list(re.finditer(r'HoraTeor=', row['Parametros']))
    
    if len(matches) >= 4:  # Verificar que haya al menos 3 coincidencias
        start = matches[3].end()  # Posición después de la tercera coincidencia
        return row['Parametros'][start:start+9]  # Extraer 9 caracteres
    
    return None

# Aplicar la función a la columna 'Parametros' y crear la nueva columna 'HoraFinTeor'
acciones['HoraFinTeor4'] = acciones.apply(extract_viaje, axis=1)

# Limpiar la columna eliminando comillas dobles y espacios en blanco
acciones['HoraFinTeor4'] = acciones['HoraFinTeor4'].str.replace('"', '').str.strip()

acciones.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Posicion,Viaje Ini,Posicion1,Viaje Ini1,Viaje_fin,Viaje_fin1,HoraTeorIni,HoraFinTeor2,HoraFinTeor3,HoraFinTeor4
0,2025-05-28,10:41:08,1091,40,Z50-4411,504411,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,200,1,126,1,2,2,10:34:18,10:39:18,12:01:00,12:06:00
1,2025-05-28,11:33:43,1091,27,Z50-4347,504347,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,199,3,125,2,4,3,11:08:58,11:13:58,13:13:30,13:18:30
2,2025-05-28,12:39:57,1182,29,Z50-4485,504485,5,Eliminar Coche,GWM,William Munar Gonzalez,...,196,6,126,5,8,7,12:43:00,12:44:00,16:56:30,17:06:30
3,2025-05-28,13:01:24,1083,1,Z50-7077,507077,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,198,4,124,3,5,4,12:50:10,12:55:10,17:20:30,17:30:30
4,2025-05-28,13:32:34,1215,6,Z50-4259,504259,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,195,6,125,9,7,11,13:00:00,13:05:00,15:00:15,15:10:15


In [ ]:
# Crear la nueva columna 'HoraTeorFin' con la prioridad establecida
acciones['HoraTeorFin'] = acciones['HoraFinTeor4'].fillna(
    acciones['HoraFinTeor3'].fillna(
        acciones['HoraFinTeor2'].fillna(
            acciones['HoraTeorIni']
        )
    )
)

acciones.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Viaje Ini,Posicion1,Viaje Ini1,Viaje_fin,Viaje_fin1,HoraTeorIni,HoraFinTeor2,HoraFinTeor3,HoraFinTeor4,HoraTeorFin
0,2025-05-28,10:41:08,1091,40,Z50-4411,504411,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,1,126,1,2,2,10:34:18,10:39:18,12:01:00,12:06:00,12:06:00
1,2025-05-28,11:33:43,1091,27,Z50-4347,504347,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,3,125,2,4,3,11:08:58,11:13:58,13:13:30,13:18:30,13:18:30
2,2025-05-28,12:39:57,1182,29,Z50-4485,504485,5,Eliminar Coche,GWM,William Munar Gonzalez,...,6,126,5,8,7,12:43:00,12:44:00,16:56:30,17:06:30,17:06:30
3,2025-05-28,13:01:24,1083,1,Z50-7077,507077,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,4,124,3,5,4,12:50:10,12:55:10,17:20:30,17:30:30,17:30:30
4,2025-05-28,13:32:34,1215,6,Z50-4259,504259,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,6,125,9,7,11,13:00:00,13:05:00,15:00:15,15:10:15,15:10:15


Accion de cambiar coche

In [ ]:
acciones_rev5 = acciones_rev.loc[acciones_rev['Accion'] == 36]

acciones_rev5

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo
299,28/05/2025,6:46:02,1594,4,Z50-4240,504240,36,Cambiar Coche,ECM,Esteban Camilo Medina,GMZ06,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje
348,28/05/2025,7:23:14,1103,9,Z50-2089,502089,36,Cambiar Coche,PAE,Edith Patarroyo,GMZ05,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje
365,28/05/2025,7:41:33,1182,22,Z50-4241,504241,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,GMZ04,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje
366,28/05/2025,7:42:36,1300,8,Z50-7099,507099,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,GMZ04,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje
390,28/05/2025,7:53:02,1083,5,Z50-7063,507063,36,Cambiar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1127,28/05/2025,14:03:23,1182,41,Z50-4310,504310,36,Cambiar Coche,BRGU,Bryan Guzman,GMZ04,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje
1128,28/05/2025,14:03:34,1208,2,Z50-2104,502104,36,Cambiar Coche,jespinosa1,John Jairo Espinosa Santa,GMZ05,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje
1135,28/05/2025,14:07:03,1030,16,Z50-4508,504508,36,Cambiar Coche,jruiz1,Jessica Briyith Espejo Ruiz,GMZ02,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje
1138,28/05/2025,14:09:59,1030,16,Z50-4508,504508,36,Cambiar Coche,MDE,Elicia Mongui Duarte,GMZ08,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje


In [ ]:
# Llevar ruta_sae a eliminaciones Motivo 31

def calcular_lin(svb):
    
    filtro = (
        (rutas['linea'] == svb) 
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not rutas.loc[filtro].empty:
        # Obtener el primer valor
        tipo = rutas.loc[filtro, 'ruta_sae'].iloc[0]
        return tipo if not pd.isna(tipo) else None 
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
acciones_rev5['ruta_sae'] = acciones_rev5.apply(
    lambda row: calcular_lin(
        row['Linea']
    ),
    axis=1
)

acciones_rev5

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9408\2959893178.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_rev5['ruta_sae'] = acciones_rev5.apply(


,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae
299,28/05/2025,6:46:02,1594,4,Z50-4240,504240,36,Cambiar Coche,ECM,Esteban Camilo Medina,GMZ06,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,7904
348,28/05/2025,7:23:14,1103,9,Z50-2089,502089,36,Cambiar Coche,PAE,Edith Patarroyo,GMZ05,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,7946
365,28/05/2025,7:41:33,1182,22,Z50-4241,504241,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,GMZ04,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,8411
366,28/05/2025,7:42:36,1300,8,Z50-7099,507099,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,GMZ04,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,5658
390,28/05/2025,7:53:02,1083,5,Z50-7063,507063,36,Cambiar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,4788
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1127,28/05/2025,14:03:23,1182,41,Z50-4310,504310,36,Cambiar Coche,BRGU,Bryan Guzman,GMZ04,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,8411
1128,28/05/2025,14:03:34,1208,2,Z50-2104,502104,36,Cambiar Coche,jespinosa1,John Jairo Espinosa Santa,GMZ05,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,8452
1135,28/05/2025,14:07:03,1030,16,Z50-4508,504508,36,Cambiar Coche,jruiz1,Jessica Briyith Espejo Ruiz,GMZ02,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,4509
1138,28/05/2025,14:09:59,1030,16,Z50-4508,504508,36,Cambiar Coche,MDE,Elicia Mongui Duarte,GMZ08,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,4509


In [ ]:
# Rellenar los valores vacíos (NaN) en la columna 'ruta_sae' con 1
acciones_rev5['ruta_sae'] = acciones_rev5['ruta_sae'].fillna(1)

acciones_rev5['ruta_sae'] = acciones_rev5['ruta_sae'].astype(int)

# Eliminar las filas donde 'ruta_sae' sea igual a 1
acciones_rev5 = acciones_rev5[acciones_rev5['ruta_sae'] != 1]

# Verificar los cambios
acciones_rev5.head()

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9408\1949342013.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_rev5['ruta_sae'] = acciones_rev5['ruta_sae'].fillna(1)
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9408\1949342013.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  acciones_rev5['ruta_sae'] = acciones_rev5['ruta_sae'].astype(int)


,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae
299,28/05/2025,6:46:02,1594,4,Z50-4240,504240,36,Cambiar Coche,ECM,Esteban Camilo Medina,GMZ06,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,7904
348,28/05/2025,7:23:14,1103,9,Z50-2089,502089,36,Cambiar Coche,PAE,Edith Patarroyo,GMZ05,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,7946
365,28/05/2025,7:41:33,1182,22,Z50-4241,504241,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,GMZ04,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,8411
366,28/05/2025,7:42:36,1300,8,Z50-7099,507099,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,GMZ04,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,5658
390,28/05/2025,7:53:02,1083,5,Z50-7063,507063,36,Cambiar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,4788


In [ ]:
#Buscar el servicio de referencia

subcadena_buscar = "VehAnterior Servicio="

# desde la posición donde se encuentra "VehAnterior Servicio="
acciones_rev5['Servicio de Referencia'] = acciones_rev5['Parametros'].str.extract(f'{subcadena_buscar}(.{{10}})')[0]

# Ahora, df['Nueva Columna'] contiene la subcadena de 9 caracteres después de "VehAnterior Servicio="

# columna de tipo cadena (string)
acciones_rev5['Servicio de Referencia'] = acciones_rev5['Servicio de Referencia'].astype(str)

# Eliminar las comillas de la columna
acciones_rev5['Servicio de Referencia'] = acciones_rev5['Servicio de Referencia'].str.replace('"', '')

acciones_rev5.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae,Servicio de Referencia
299,28/05/2025,6:46:02,1594,4,Z50-4240,504240,36,Cambiar Coche,ECM,Esteban Camilo Medina,GMZ06,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,7904,CEID00004
348,28/05/2025,7:23:14,1103,9,Z50-2089,502089,36,Cambiar Coche,PAE,Edith Patarroyo,GMZ05,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,7946,CEIH70009
365,28/05/2025,7:41:33,1182,22,Z50-4241,504241,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,GMZ04,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,8411,CEIJW0015
366,28/05/2025,7:42:36,1300,8,Z50-7099,507099,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,GMZ04,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,5658,CNLRX0011
390,28/05/2025,7:53:02,1083,5,Z50-7063,507063,36,Cambiar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,4788,CEIKB0005


In [ ]:
# Encontrar la posición de "Viaje=" en la columna 'Parámetros'
acciones_rev5['Posicion'] = acciones_rev5['Parametros'].str.find("Viaje=")

# Extraer los últimos 2 caracteres de la subcadena a partir de la posición encontrada
acciones_rev5['Viaje Ini'] = acciones_rev5.apply(lambda row: row['Parametros'][row['Posicion']+6:row['Posicion']+9], axis=1)

# Reemplazar las comillas simples (') por un valor en blanco
acciones_rev5['Viaje Ini'] = acciones_rev5['Viaje Ini'].str.replace('"', '')

# Eliminar espacios en blanco al principio o al final de la cadena
acciones_rev5['Viaje Ini'] = acciones_rev5['Viaje Ini'].str.strip()

acciones_rev5.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae,Servicio de Referencia,Posicion,Viaje Ini
299,28/05/2025,6:46:02,1594,4,Z50-4240,504240,36,Cambiar Coche,ECM,Esteban Camilo Medina,GMZ06,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,7904,CEID00004,251,3
348,28/05/2025,7:23:14,1103,9,Z50-2089,502089,36,Cambiar Coche,PAE,Edith Patarroyo,GMZ05,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,7946,CEIH70009,251,3
365,28/05/2025,7:41:33,1182,22,Z50-4241,504241,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,GMZ04,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,8411,CEIJW0015,251,4
366,28/05/2025,7:42:36,1300,8,Z50-7099,507099,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,GMZ04,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,5658,CNLRX0011,251,3
390,28/05/2025,7:53:02,1083,5,Z50-7063,507063,36,Cambiar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,4788,CEIKB0005,251,3


In [ ]:
# Encontrar la posición de "Viaje=" en la columna 'Parámetros'
acciones_rev5['Posicion'] = acciones_rev5['Parametros'].str.find("HoraIniTeor=")

# Extraer los últimos 2 caracteres de la subcadena a partir de la posición encontrada
acciones_rev5['HoraIniTeor'] = acciones_rev5.apply(lambda row: row['Parametros'][row['Posicion']+12:row['Posicion']+21], axis=1)

# Reemplazar las comillas simples (') por un valor en blanco
acciones_rev5['HoraIniTeor'] = acciones_rev5['HoraIniTeor'].str.replace('"', '')

# Eliminar espacios en blanco al principio o al final de la cadena
acciones_rev5['HoraIniTeor'] = acciones_rev5['HoraIniTeor'].str.strip()

acciones_rev5.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae,Servicio de Referencia,Posicion,Viaje Ini,HoraIniTeor
299,28/05/2025,6:46:02,1594,4,Z50-4240,504240,36,Cambiar Coche,ECM,Esteban Camilo Medina,GMZ06,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,7904,CEID00004,324,3,06:51:00
348,28/05/2025,7:23:14,1103,9,Z50-2089,502089,36,Cambiar Coche,PAE,Edith Patarroyo,GMZ05,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,7946,CEIH70009,324,3,07:26:30
365,28/05/2025,7:41:33,1182,22,Z50-4241,504241,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,GMZ04,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,8411,CEIJW0015,324,4,07:36:00
366,28/05/2025,7:42:36,1300,8,Z50-7099,507099,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,GMZ04,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,5658,CNLRX0011,324,3,07:38:00
390,28/05/2025,7:53:02,1083,5,Z50-7063,507063,36,Cambiar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,4788,CEIKB0005,324,3,07:53:45


In [ ]:
# Función para extraer 2 caracteres después de la segunda aparición de "Viaje="
def extract_viaje(row):
    matches = re.finditer(r'Viaje=', row['Parametros'])
    # Encontrar la segunda coincidencia
    match = next(matches, None)
    if match:
        match = next(matches, None)  # Segunda coincidencia
        if match:
            start = match.end()  # Posición de inicio después de la segunda coincidencia
            return row['Parametros'][start:start+4]  # Extraer 4 caracteres
    return None

# Aplicar la función a la columna Parametros y crear una nueva columna 'Viaje_extraido'
acciones_rev5['Viaje_fin'] = acciones_rev5.apply(extract_viaje, axis=1)

# Reemplazar las comillas simples (') por un valor en blanco
acciones_rev5['Viaje_fin'] = acciones_rev5['Viaje_fin'].str.replace('"', '')

# Eliminar espacios en blanco al principio o al final de la cadena
acciones_rev5['Viaje_fin'] = acciones_rev5['Viaje_fin'].str.strip()

acciones_rev5.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae,Servicio de Referencia,Posicion,Viaje Ini,HoraIniTeor,Viaje_fin
299,28/05/2025,6:46:02,1594,4,Z50-4240,504240,36,Cambiar Coche,ECM,Esteban Camilo Medina,GMZ06,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,7904,CEID00004,324,3,06:51:00,8
348,28/05/2025,7:23:14,1103,9,Z50-2089,502089,36,Cambiar Coche,PAE,Edith Patarroyo,GMZ05,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,7946,CEIH70009,324,3,07:26:30,7
365,28/05/2025,7:41:33,1182,22,Z50-4241,504241,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,GMZ04,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,8411,CEIJW0015,324,4,07:36:00,10
366,28/05/2025,7:42:36,1300,8,Z50-7099,507099,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,GMZ04,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,5658,CNLRX0011,324,3,07:38:00,8
390,28/05/2025,7:53:02,1083,5,Z50-7063,507063,36,Cambiar Coche,BEJA,Jenny Alexandra Bernal,GMZ03,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,4788,CEIKB0005,324,3,07:53:45,5


In [ ]:
# Función para extraer 2 caracteres después de la segunda aparición de "Viaje="
def extract_viaje(row):
    matches = re.finditer(r'HoraFinTeor=', row['Parametros'])
    # Encontrar la segunda coincidencia
    match = next(matches, None)
    if match:
        match = next(matches, None)  # Segunda coincidencia
        if match:
            start = match.end()  # Posición de inicio después de la segunda coincidencia
            return row['Parametros'][start:start+10]  # Extraer 4 caracteres
    return None

# Aplicar la función a la columna Parametros y crear una nueva columna 'Viaje_extraido'
acciones_rev5['HoraFinTeor'] = acciones_rev5.apply(extract_viaje, axis=1)

# Reemplazar las comillas simples (') por un valor en blanco
acciones_rev5['HoraFinTeor'] = acciones_rev5['HoraFinTeor'].str.replace('"', '')

# Eliminar espacios en blanco al principio o al final de la cadena
acciones_rev5['HoraFinTeor'] = acciones_rev5['HoraFinTeor'].str.strip()

acciones_rev5.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Parametros,Motivo,DescripcionMotivo,ruta_sae,Servicio de Referencia,Posicion,Viaje Ini,HoraIniTeor,Viaje_fin,HoraFinTeor
299,28/05/2025,6:46:02,1594,4,Z50-4240,504240,36,Cambiar Coche,ECM,Esteban Camilo Medina,...,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,7904,CEID00004,324,3,06:51:00,8,23:49:30
348,28/05/2025,7:23:14,1103,9,Z50-2089,502089,36,Cambiar Coche,PAE,Edith Patarroyo,...,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,7946,CEIH70009,324,3,07:26:30,7,23:41:30
365,28/05/2025,7:41:33,1182,22,Z50-4241,504241,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,...,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,8411,CEIJW0015,324,4,07:36:00,10,22:28:45
366,28/05/2025,7:42:36,1300,8,Z50-7099,507099,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,...,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,5658,CNLRX0011,324,3,07:38:00,8,23:52:00
390,28/05/2025,7:53:02,1083,5,Z50-7063,507063,36,Cambiar Coche,BEJA,Jenny Alexandra Bernal,...,"<CambiarBus Motivo=""26""><VehAnterior Servicio=...",26,Retoma de viaje,4788,CEIKB0005,324,3,07:53:45,5,22:31:45


In [ ]:
#Buscar el servicio de referencia

subcadena_buscar = "VehNuevo Servicio="

# desde la posición donde se encuentra "VehAnterior Servicio="
acciones_rev5['Nombre Servicio'] = acciones_rev5['Parametros'].str.extract(f'{subcadena_buscar}(.{{10}})')[0]

# Ahora, df['Nueva Columna'] contiene la subcadena de 9 caracteres después de "VehAnterior Servicio="

# columna de tipo cadena (string)
acciones_rev5['Nombre Servicio'] = acciones_rev5['Nombre Servicio'].astype(str)

# Eliminar las comillas de la columna
acciones_rev5['Nombre Servicio'] = acciones_rev5['Nombre Servicio'].str.replace('"', '')

# Eliminar espacios en blanco al principio o al final de la cadena
acciones_rev5['Nombre Servicio'] = acciones_rev5['Nombre Servicio'].str.strip()

acciones_rev5.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Motivo,DescripcionMotivo,ruta_sae,Servicio de Referencia,Posicion,Viaje Ini,HoraIniTeor,Viaje_fin,HoraFinTeor,Nombre Servicio
299,28/05/2025,6:46:02,1594,4,Z50-4240,504240,36,Cambiar Coche,ECM,Esteban Camilo Medina,...,26,Retoma de viaje,7904,CEID00004,324,3,06:51:00,8,23:49:30,CEID0G004
348,28/05/2025,7:23:14,1103,9,Z50-2089,502089,36,Cambiar Coche,PAE,Edith Patarroyo,...,26,Retoma de viaje,7946,CEIH70009,324,3,07:26:30,7,23:41:30,CEIH7G009
365,28/05/2025,7:41:33,1182,22,Z50-4241,504241,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,...,26,Retoma de viaje,8411,CEIJW0015,324,4,07:36:00,10,22:28:45,CEIJWG015
366,28/05/2025,7:42:36,1300,8,Z50-7099,507099,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,...,26,Retoma de viaje,5658,CNLRX0011,324,3,07:38:00,8,23:52:00,CNLRXG011
390,28/05/2025,7:53:02,1083,5,Z50-7063,507063,36,Cambiar Coche,BEJA,Jenny Alexandra Bernal,...,26,Retoma de viaje,4788,CEIKB0005,324,3,07:53:45,5,22:31:45,CEIKBG005


In [ ]:
# Susituyente de sustituyente

acciones_rev5['Sustituyente de Sustituyente'] = acciones_rev5.apply(lambda row: 'Si' if row['Nombre Servicio'] == row['Servicio de Referencia'] else 'No', axis=1)

# Creando servicio original

acciones_rev5['Servicio Planificado'] = acciones_rev5['Nombre Servicio'].str[:5] + '0' + acciones_rev5['Nombre Servicio'].str[-3:]

# Crear una nueva columna 'Conteo' 

acciones_rev5['Numero de sustituyente'] = acciones_rev5.apply(lambda row: 
    len(acciones_rev5[(acciones_rev5['Servicio de Referencia'] < row['Servicio de Referencia']) & 
                    (acciones_rev5['Servicio Planificado'] == row['Servicio Planificado'])]) + 1, axis=1)

acciones_rev5.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Servicio de Referencia,Posicion,Viaje Ini,HoraIniTeor,Viaje_fin,HoraFinTeor,Nombre Servicio,Sustituyente de Sustituyente,Servicio Planificado,Numero de sustituyente
299,28/05/2025,6:46:02,1594,4,Z50-4240,504240,36,Cambiar Coche,ECM,Esteban Camilo Medina,...,CEID00004,324,3,06:51:00,8,23:49:30,CEID0G004,No,CEID00004,1
348,28/05/2025,7:23:14,1103,9,Z50-2089,502089,36,Cambiar Coche,PAE,Edith Patarroyo,...,CEIH70009,324,3,07:26:30,7,23:41:30,CEIH7G009,No,CEIH70009,1
365,28/05/2025,7:41:33,1182,22,Z50-4241,504241,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,...,CEIJW0015,324,4,07:36:00,10,22:28:45,CEIJWG015,No,CEIJW0015,1
366,28/05/2025,7:42:36,1300,8,Z50-7099,507099,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,...,CNLRX0011,324,3,07:38:00,8,23:52:00,CNLRXG011,No,CNLRX0011,1
390,28/05/2025,7:53:02,1083,5,Z50-7063,507063,36,Cambiar Coche,BEJA,Jenny Alexandra Bernal,...,CEIKB0005,324,3,07:53:45,5,22:31:45,CEIKBG005,No,CEIKB0005,1


In [ ]:
#N Viajes

# Convierte las columnas 'Viaje Fin' y 'Viaje Ini' a tipo numérico (int)
acciones_rev5['Viaje_fin'] = acciones_rev5['Viaje_fin'].astype(int)
acciones_rev5['Viaje Ini'] = acciones_rev5['Viaje Ini'].astype(int)


def calcular_resultado(row):
    if row['Sustituyente de Sustituyente'] == "No" and \
        row['Numero de sustituyente'] >= 1 and \
        row['Viaje_fin'] > row['Viaje Ini']:
        return row['Viaje_fin'] - row['Viaje Ini']
    
    if row['Sustituyente de Sustituyente'] == "Si":
        viaje_ini_lookup = acciones_rev5[(acciones_rev5['Servicio de Referencia'] == row['Servicio de Referencia']) & (acciones_rev5['Nombre Servicio'] == row['Nombre Servicio'])]['Viaje Ini'].values
        if len(viaje_ini_lookup) == 0:
            return 1 if row['Viaje_fin'] == row['Viaje Ini'] else row['Viaje_fin'] - row['Viaje Ini']
        
        if row['Numero de sustituyente'] == 1:
            return row['Viaje_fin'] - row['Viaje Ini'] + 1
        
        viaje_ini_anterior = acciones_rev5[(acciones_rev5['Servicio Planificado'] == row['Servicio Planificado']) & (acciones_rev5['Numero de sustiituyente'] == row['Numero de sustituyente'] - 1)]['Viaje Ini'].values
        if len(viaje_ini_anterior) > 0:
            return viaje_ini_lookup[0] - row['Viaje Ini'] - 1
        
    if row['Sustituyente de Sustituyente'] == "No" and row['Viaje_fin'] == row['Viaje Ini']:
        return 1
    
    return None

acciones_rev5['N Viajes'] = acciones_rev5.apply(calcular_resultado, axis=1)

acciones_rev5

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Posicion,Viaje Ini,HoraIniTeor,Viaje_fin,HoraFinTeor,Nombre Servicio,Sustituyente de Sustituyente,Servicio Planificado,Numero de sustituyente,N Viajes
299,28/05/2025,6:46:02,1594,4,Z50-4240,504240,36,Cambiar Coche,ECM,Esteban Camilo Medina,...,324,3,06:51:00,8,23:49:30,CEID0G004,No,CEID00004,1,5
348,28/05/2025,7:23:14,1103,9,Z50-2089,502089,36,Cambiar Coche,PAE,Edith Patarroyo,...,324,3,07:26:30,7,23:41:30,CEIH7G009,No,CEIH70009,1,4
365,28/05/2025,7:41:33,1182,22,Z50-4241,504241,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,...,324,4,07:36:00,10,22:28:45,CEIJWG015,No,CEIJW0015,1,6
366,28/05/2025,7:42:36,1300,8,Z50-7099,507099,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,...,324,3,07:38:00,8,23:52:00,CNLRXG011,No,CNLRX0011,1,5
390,28/05/2025,7:53:02,1083,5,Z50-7063,507063,36,Cambiar Coche,BEJA,Jenny Alexandra Bernal,...,324,3,07:53:45,5,22:31:45,CEIKBG005,No,CEIKB0005,1,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1127,28/05/2025,14:03:23,1182,41,Z50-4310,504310,36,Cambiar Coche,BRGU,Bryan Guzman,...,324,4,13:58:30,7,22:28:45,CEIJWH015,No,CEIJW0015,2,3
1128,28/05/2025,14:03:34,1208,2,Z50-2104,502104,36,Cambiar Coche,jespinosa1,John Jairo Espinosa Santa,...,324,6,14:18:00,7,20:09:30,CEIKAG002,No,CEIKA0002,1,1
1135,28/05/2025,14:07:03,1030,16,Z50-4508,504508,36,Cambiar Coche,jruiz1,Jessica Briyith Espejo Ruiz,...,324,5,19:08:30,5,23:04:00,CEICVG016,No,CEICV0016,1,1
1138,28/05/2025,14:09:59,1030,16,Z50-4508,504508,36,Cambiar Coche,MDE,Elicia Mongui Duarte,...,324,4,13:56:45,4,19:06:45,CEICVH016,No,CEICV0016,1,1


Accion de retoma

In [ ]:
acciones_rev4 = acciones_rev.copy()

acciones_rev4 = acciones_rev4.loc[acciones_rev4['Accion'] == 4]

acciones_rev4

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo
431,28/05/2025,8:16:03,1182,43,NaN,0,4,Introducir Coche,GWM,William Munar Gonzalez,GMZ04,"<CrearCoche Servicio=""RTB822503"" ServicioCondu...",26,Retoma de viaje
629,28/05/2025,9:51:24,1031,33,NaN,0,4,Introducir Coche,VSC,Sandra Carolina Valeriano,GMZ08,"<CrearCoche Servicio=""RTA310803"" ServicioCondu...",31,Viaje Finalizado


In [ ]:
# Llevar ruta_sae a eliminaciones Motivo 31

def calcular_lin(svb):
    
    filtro = (
        (rutas['linea'] == svb) 
    )
    
    # Verificar si hay algún resultado después de aplicar el filtro
    if not rutas.loc[filtro].empty:
        # Obtener el primer valor
        tipo = rutas.loc[filtro, 'ruta_sae'].iloc[0]
        return tipo if not pd.isna(tipo) else None 
    
    return None  # Devolver None si no hay resultados

# Aplica la función a las columnas correspondientes
acciones_rev4['ruta_sae'] = acciones_rev4.apply(
    lambda row: calcular_lin(
        row['Linea']
    ),
    axis=1
)

acciones_rev4

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae
431,28/05/2025,8:16:03,1182,43,NaN,0,4,Introducir Coche,GWM,William Munar Gonzalez,GMZ04,"<CrearCoche Servicio=""RTB822503"" ServicioCondu...",26,Retoma de viaje,8411
629,28/05/2025,9:51:24,1031,33,NaN,0,4,Introducir Coche,VSC,Sandra Carolina Valeriano,GMZ08,"<CrearCoche Servicio=""RTA310803"" ServicioCondu...",31,Viaje Finalizado,4628


In [ ]:
# Rellenar los valores vacíos (NaN) en la columna 'ruta_sae' con 1
acciones_rev4['ruta_sae'] = acciones_rev4['ruta_sae'].fillna(1)

acciones_rev4['ruta_sae'] = acciones_rev4['ruta_sae'].astype(int)

# Eliminar las filas donde 'ruta_sae' sea igual a 1
acciones_rev4 = acciones_rev4[acciones_rev4['ruta_sae'] != 1]

# Verificar los cambios
acciones_rev4.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae
431,28/05/2025,8:16:03,1182,43,NaN,0,4,Introducir Coche,GWM,William Munar Gonzalez,GMZ04,"<CrearCoche Servicio=""RTB822503"" ServicioCondu...",26,Retoma de viaje,8411
629,28/05/2025,9:51:24,1031,33,NaN,0,4,Introducir Coche,VSC,Sandra Carolina Valeriano,GMZ08,"<CrearCoche Servicio=""RTA310803"" ServicioCondu...",31,Viaje Finalizado,4628


In [ ]:
#Buscar el servicio de referencia

subcadena_buscar = "Referencia Servicio="

# desde la posición donde se encuentra "VehAnterior Servicio="
acciones_rev4['Servicio de Referencia'] = acciones_rev4['Parametros'].str.extract(f'{subcadena_buscar}(.{{10}})')[0]

# Ahora, df['Nueva Columna'] contiene la subcadena de 9 caracteres después de "VehAnterior Servicio="

# columna de tipo cadena (string)
acciones_rev4['Servicio de Referencia'] = acciones_rev4['Servicio de Referencia'].astype(str)

# Eliminar las comillas de la columna
acciones_rev4['Servicio de Referencia'] = acciones_rev4['Servicio de Referencia'].str.replace('"', '')

acciones_rev4.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae,Servicio de Referencia
431,28/05/2025,8:16:03,1182,43,NaN,0,4,Introducir Coche,GWM,William Munar Gonzalez,GMZ04,"<CrearCoche Servicio=""RTB822503"" ServicioCondu...",26,Retoma de viaje,8411,CEIJW0023
629,28/05/2025,9:51:24,1031,33,NaN,0,4,Introducir Coche,VSC,Sandra Carolina Valeriano,GMZ08,"<CrearCoche Servicio=""RTA310803"" ServicioCondu...",31,Viaje Finalizado,4628,CEIH50010


In [ ]:
# Encontrar la posición de "Viaje=" en la columna 'Parámetros'
acciones_rev4['Posicion'] = acciones_rev4['Parametros'].str.find("IdViaje=")

# Extraer los últimos 2 caracteres de la subcadena a partir de la posición encontrada
acciones_rev4['Viaje Ini'] = acciones_rev4.apply(lambda row: row['Parametros'][row['Posicion']+8:row['Posicion']+11], axis=1)

# Reemplazar las comillas simples (') por un valor en blanco
acciones_rev4['Viaje Ini'] = acciones_rev4['Viaje Ini'].str.replace('"', '')

# Eliminar espacios en blanco al principio o al final de la cadena
acciones_rev4['Viaje Ini'] = acciones_rev4['Viaje Ini'].str.strip()

acciones_rev4.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae,Servicio de Referencia,Posicion,Viaje Ini
431,28/05/2025,8:16:03,1182,43,NaN,0,4,Introducir Coche,GWM,William Munar Gonzalez,GMZ04,"<CrearCoche Servicio=""RTB822503"" ServicioCondu...",26,Retoma de viaje,8411,CEIJW0023,378,4
629,28/05/2025,9:51:24,1031,33,NaN,0,4,Introducir Coche,VSC,Sandra Carolina Valeriano,GMZ08,"<CrearCoche Servicio=""RTA310803"" ServicioCondu...",31,Viaje Finalizado,4628,CEIH50010,377,4


In [ ]:
# Encontrar la posición de "Viaje=" en la columna 'Parámetros'
acciones_rev4['Posicion1'] = acciones_rev4['Parametros'].str.find("ViajeLinea=")

# Extraer los últimos 2 caracteres de la subcadena a partir de la posición encontrada
acciones_rev4['Viaje Ini1'] = acciones_rev4.apply(lambda row: row['Parametros'][row['Posicion1']+11:row['Posicion1']+14], axis=1)

# Reemplazar las comillas simples (') por un valor en blanco
acciones_rev4['Viaje Ini1'] = acciones_rev4['Viaje Ini1'].str.replace('"', '')

# Eliminar espacios en blanco al principio o al final de la cadena
acciones_rev4['Viaje Ini1'] = acciones_rev4['Viaje Ini1'].str.strip()

acciones_rev4.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,Puesto,Parametros,Motivo,DescripcionMotivo,ruta_sae,Servicio de Referencia,Posicion,Viaje Ini,Posicion1,Viaje Ini1
431,28/05/2025,8:16:03,1182,43,NaN,0,4,Introducir Coche,GWM,William Munar Gonzalez,GMZ04,"<CrearCoche Servicio=""RTB822503"" ServicioCondu...",26,Retoma de viaje,8411,CEIJW0023,378,4,390,3
629,28/05/2025,9:51:24,1031,33,NaN,0,4,Introducir Coche,VSC,Sandra Carolina Valeriano,GMZ08,"<CrearCoche Servicio=""RTA310803"" ServicioCondu...",31,Viaje Finalizado,4628,CEIH50010,377,4,389,3


In [ ]:
#Buscar el servicio

subcadena_buscar = "Servicio="

# desde la posición donde se encuentra "VehAnterior Servicio="
acciones_rev4['Nombre Servicio'] = acciones_rev4['Parametros'].str.extract(f'{subcadena_buscar}(.{{10}})')[0]

# Ahora, df['Nueva Columna'] contiene la subcadena de 9 caracteres después de "VehAnterior Servicio="

# columna de tipo cadena (string)
acciones_rev4['Nombre Servicio'] = acciones_rev4['Nombre Servicio'].astype(str)

# Eliminar las comillas de la columna
acciones_rev4['Nombre Servicio'] = acciones_rev4['Nombre Servicio'].str.replace('"', '')

acciones_rev4

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Parametros,Motivo,DescripcionMotivo,ruta_sae,Servicio de Referencia,Posicion,Viaje Ini,Posicion1,Viaje Ini1,Nombre Servicio
431,28/05/2025,8:16:03,1182,43,NaN,0,4,Introducir Coche,GWM,William Munar Gonzalez,...,"<CrearCoche Servicio=""RTB822503"" ServicioCondu...",26,Retoma de viaje,8411,CEIJW0023,378,4,390,3,RTB822503
629,28/05/2025,9:51:24,1031,33,NaN,0,4,Introducir Coche,VSC,Sandra Carolina Valeriano,...,"<CrearCoche Servicio=""RTA310803"" ServicioCondu...",31,Viaje Finalizado,4628,CEIH50010,377,4,389,3,RTA310803


In [ ]:
# Encontrar la posición de "Viaje=" en la columna 'Parámetros'
acciones_rev4['Posicion'] = acciones_rev4['Parametros'].str.find("IdViaje=")

# Extraer los últimos 2 caracteres de la subcadena a partir de la posición encontrada
acciones_rev4['Viaje Ini'] = acciones_rev4.apply(lambda row: row['Parametros'][row['Posicion']+8:row['Posicion']+11], axis=1)

# Reemplazar las comillas simples (') por un valor en blanco
acciones_rev4['Viaje Ini'] = acciones_rev4['Viaje Ini'].str.replace('"', '')

# Eliminar espacios en blanco al principio o al final de la cadena
acciones_rev4['Viaje Ini'] = acciones_rev4['Viaje Ini'].str.strip()

acciones_rev4.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Parametros,Motivo,DescripcionMotivo,ruta_sae,Servicio de Referencia,Posicion,Viaje Ini,Posicion1,Viaje Ini1,Nombre Servicio
431,28/05/2025,8:16:03,1182,43,NaN,0,4,Introducir Coche,GWM,William Munar Gonzalez,...,"<CrearCoche Servicio=""RTB822503"" ServicioCondu...",26,Retoma de viaje,8411,CEIJW0023,378,4,390,3,RTB822503
629,28/05/2025,9:51:24,1031,33,NaN,0,4,Introducir Coche,VSC,Sandra Carolina Valeriano,...,"<CrearCoche Servicio=""RTA310803"" ServicioCondu...",31,Viaje Finalizado,4628,CEIH50010,377,4,389,3,RTA310803


In [ ]:
# Función para extraer 2 caracteres después de la segunda aparición de "Viaje="
def extract_viaje(row):
    matches = re.finditer(r'HoraIniTeor=', row['Parametros'])
    # Encontrar la segunda coincidencia
    match = next(matches, None)
    if match:
        match = next(matches, None)  # Segunda coincidencia
        if match:
            start = match.end()  # Posición de inicio después de la segunda coincidencia
            return row['Parametros'][start:start+9]  # Extraer 4 caracteres
    return None

# Aplicar la función a la columna Parametros y crear una nueva columna 'Viaje_extraido'
acciones_rev4['HoraIniTeor'] = acciones_rev4.apply(extract_viaje, axis=1)

# Reemplazar las comillas simples (') por un valor en blanco
acciones_rev4['HoraIniTeor'] = acciones_rev4['HoraIniTeor'].str.replace('"', '')

# Eliminar espacios en blanco al principio o al final de la cadena
acciones_rev4['HoraIniTeor'] = acciones_rev4['HoraIniTeor'].str.strip()

acciones_rev4

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Motivo,DescripcionMotivo,ruta_sae,Servicio de Referencia,Posicion,Viaje Ini,Posicion1,Viaje Ini1,Nombre Servicio,HoraIniTeor
431,28/05/2025,8:16:03,1182,43,NaN,0,4,Introducir Coche,GWM,William Munar Gonzalez,...,26,Retoma de viaje,8411,CEIJW0023,378,4,390,3,RTB822503,08:06:00
629,28/05/2025,9:51:24,1031,33,NaN,0,4,Introducir Coche,VSC,Sandra Carolina Valeriano,...,31,Viaje Finalizado,4628,CEIH50010,377,4,389,3,RTA310803,10:14:45


In [ ]:
# Función para extraer 2 caracteres después de la segunda aparición de "Viaje="
def extract_viaje(row):
    matches = re.finditer(r'IdViaje=', row['Parametros'])
    # Encontrar la segunda coincidencia
    match = next(matches, None)
    if match:
        match = next(matches, None)  # Segunda coincidencia
        if match:
            start = match.end()  # Posición de inicio después de la segunda coincidencia
            return row['Parametros'][start:start+4]  # Extraer 4 caracteres
    return None

# Aplicar la función a la columna Parametros y crear una nueva columna 'Viaje_extraido'
acciones_rev4['Viaje_fin'] = acciones_rev4.apply(extract_viaje, axis=1)

# Reemplazar las comillas simples (') por un valor en blanco
acciones_rev4['Viaje_fin'] = acciones_rev4['Viaje_fin'].str.replace('"', '')

# Eliminar espacios en blanco al principio o al final de la cadena
acciones_rev4['Viaje_fin'] = acciones_rev4['Viaje_fin'].str.strip()

acciones_rev4

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,DescripcionMotivo,ruta_sae,Servicio de Referencia,Posicion,Viaje Ini,Posicion1,Viaje Ini1,Nombre Servicio,HoraIniTeor,Viaje_fin
431,28/05/2025,8:16:03,1182,43,NaN,0,4,Introducir Coche,GWM,William Munar Gonzalez,...,Retoma de viaje,8411,CEIJW0023,378,4,390,3,RTB822503,08:06:00,4
629,28/05/2025,9:51:24,1031,33,NaN,0,4,Introducir Coche,VSC,Sandra Carolina Valeriano,...,Viaje Finalizado,4628,CEIH50010,377,4,389,3,RTA310803,10:14:45,4


In [ ]:
# Función para extraer el valor después de la tercera aparición de "HoraFinTeor="
def extract_viaje(row):
    matches = list(re.finditer(r'HoraFinTeor=', row['Parametros']))
    
    if len(matches) >= 3:  # Verificar que haya al menos 3 coincidencias
        start = matches[2].end()  # Posición después de la tercera coincidencia
        return row['Parametros'][start:start+9]  # Extraer 9 caracteres
    
    return None

# Aplicar la función a la columna 'Parametros' y crear la nueva columna 'HoraFinTeor'
acciones_rev4['HoraFinTeor'] = acciones_rev4.apply(extract_viaje, axis=1)

# Limpiar la columna eliminando comillas dobles y espacios en blanco
acciones_rev4['HoraFinTeor'] = acciones_rev4['HoraFinTeor'].str.replace('"', '').str.strip()

acciones_rev4

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,ruta_sae,Servicio de Referencia,Posicion,Viaje Ini,Posicion1,Viaje Ini1,Nombre Servicio,HoraIniTeor,Viaje_fin,HoraFinTeor
431,28/05/2025,8:16:03,1182,43,NaN,0,4,Introducir Coche,GWM,William Munar Gonzalez,...,8411,CEIJW0023,378,4,390,3,RTB822503,08:06:00,4,10:07:00
629,28/05/2025,9:51:24,1031,33,NaN,0,4,Introducir Coche,VSC,Sandra Carolina Valeriano,...,4628,CEIH50010,377,4,389,3,RTA310803,10:14:45,4,12:56:45


In [ ]:
# Función para extraer 2 caracteres después de la segunda aparición de "Viaje="
def extract_viaje(row):
    matches = re.finditer(r'ViajeLinea=', row['Parametros'])
    # Encontrar la segunda coincidencia
    match = next(matches, None)
    if match:
        match = next(matches, None)  # Segunda coincidencia
        if match:
            start = match.end()  # Posición de inicio después de la segunda coincidencia
            return row['Parametros'][start:start+4]  # Extraer 4 caracteres
    return None

# Aplicar la función a la columna Parametros y crear una nueva columna 'Viaje_extraido'
acciones_rev4['Viaje_fin1'] = acciones_rev4.apply(extract_viaje, axis=1)

# Reemplazar las comillas simples (') por un valor en blanco
acciones_rev4['Viaje_fin1'] = acciones_rev4['Viaje_fin1'].str.replace('"', '')

# Eliminar espacios en blanco al principio o al final de la cadena
acciones_rev4['Viaje_fin1'] = acciones_rev4['Viaje_fin1'].str.strip()

acciones_rev4.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Servicio de Referencia,Posicion,Viaje Ini,Posicion1,Viaje Ini1,Nombre Servicio,HoraIniTeor,Viaje_fin,HoraFinTeor,Viaje_fin1
431,28/05/2025,8:16:03,1182,43,NaN,0,4,Introducir Coche,GWM,William Munar Gonzalez,...,CEIJW0023,378,4,390,3,RTB822503,08:06:00,4,10:07:00,3
629,28/05/2025,9:51:24,1031,33,NaN,0,4,Introducir Coche,VSC,Sandra Carolina Valeriano,...,CEIH50010,377,4,389,3,RTA310803,10:14:45,4,12:56:45,3


In [ ]:
#Buscar incorporación

# Función personalizada para extraer la subcadena
def extraer_incorporacion_cochera(parametros):
    start = parametros.find("Incorporacion Cochera=")
    if start != -1:
        substring = parametros[start + len("Incorporacion Cochera="):start + len("Incorporacion Cochera=") + 9]
        return substring
    else:
        return None

# Aplicar la función personalizada a la columna 'Parámetros'
acciones_rev4['Con Incorporacion'] = acciones_rev4['Parametros'].apply(lambda x: 1 if extraer_incorporacion_cochera(x) is not None else 0)

acciones_rev4

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Posicion,Viaje Ini,Posicion1,Viaje Ini1,Nombre Servicio,HoraIniTeor,Viaje_fin,HoraFinTeor,Viaje_fin1,Con Incorporacion
431,28/05/2025,8:16:03,1182,43,NaN,0,4,Introducir Coche,GWM,William Munar Gonzalez,...,378,4,390,3,RTB822503,08:06:00,4,10:07:00,3,1
629,28/05/2025,9:51:24,1031,33,NaN,0,4,Introducir Coche,VSC,Sandra Carolina Valeriano,...,377,4,389,3,RTA310803,10:14:45,4,12:56:45,3,1


In [ ]:
#N Viajes

# Convierte las columnas 'Viaje Fin' y 'Viaje Ini' a tipo numérico (int)
acciones_rev4['Viaje_fin'] = acciones_rev4['Viaje_fin'].astype(int)
acciones_rev4['Viaje Ini'] = acciones_rev4['Viaje Ini'].astype(int)

# Primera condición: LOOKUPVALUE(Retomas[Viaje Ini]; Retomas[Servicio de Referencia]; Retomas[Nombre Servicio]) = 2
# y Retomas[Con Incorporación] = 1
condicion_1 = (acciones_rev4['Viaje Ini'] == 2) & (acciones_rev4['Con Incorporacion'] == 1)

# Si se cumple la primera condición, el resultado es 0; de lo contrario, calculamos Retomas[Viaje Fin] - Retomas[Viaje Ini] + 1
acciones_rev4['N Viajes'] = 0
acciones_rev4.loc[~condicion_1, 'N Viajes'] = acciones_rev4['Viaje_fin'] - acciones_rev4['Viaje Ini'] + 1

acciones_rev4

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Viaje Ini,Posicion1,Viaje Ini1,Nombre Servicio,HoraIniTeor,Viaje_fin,HoraFinTeor,Viaje_fin1,Con Incorporacion,N Viajes
431,28/05/2025,8:16:03,1182,43,NaN,0,4,Introducir Coche,GWM,William Munar Gonzalez,...,4,390,3,RTB822503,08:06:00,4,10:07:00,3,1,1
629,28/05/2025,9:51:24,1031,33,NaN,0,4,Introducir Coche,VSC,Sandra Carolina Valeriano,...,4,389,3,RTA310803,10:14:45,4,12:56:45,3,1,1


In [ ]:
# Filtrar solo las filas donde hay exactamente dos cifras antes del primer ':'
acciones= acciones[acciones['HoraTeorIni'].str.match(r'^\d{2}:\d{1,3}:\d{1,2}$')]

# Mostrar el resultado
acciones.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Viaje Ini,Posicion1,Viaje Ini1,Viaje_fin,Viaje_fin1,HoraTeorIni,HoraFinTeor2,HoraFinTeor3,HoraFinTeor4,HoraTeorFin
0,2025-05-28,10:41:08,1091,40,Z50-4411,504411,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,1,126,1,2,2,10:34:18,10:39:18,12:01:00,12:06:00,12:06:00
1,2025-05-28,11:33:43,1091,27,Z50-4347,504347,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,3,125,2,4,3,11:08:58,11:13:58,13:13:30,13:18:30,13:18:30
2,2025-05-28,12:39:57,1182,29,Z50-4485,504485,5,Eliminar Coche,GWM,William Munar Gonzalez,...,6,126,5,8,7,12:43:00,12:44:00,16:56:30,17:06:30,17:06:30
3,2025-05-28,13:01:24,1083,1,Z50-7077,507077,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,4,124,3,5,4,12:50:10,12:55:10,17:20:30,17:30:30,17:30:30
4,2025-05-28,13:32:34,1215,6,Z50-4259,504259,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,6,125,9,7,11,13:00:00,13:05:00,15:00:15,15:10:15,15:10:15


In [ ]:
# Función para corregir tiempos con "24:00:00"
def fix_time(date_str):
    if '24:' in date_str:
        return date_str.replace('24:', '00:')
    elif '25:' in date_str:
        return date_str.replace('25:', '01:')
    elif '26:' in date_str:
        return date_str.replace('26:', '02:')
    elif '27:' in date_str:
        return date_str.replace('27:', '03:')
    elif '28:' in date_str:
        return date_str.replace('28:', '04:')
    elif '29:' in date_str:
        return date_str.replace('29:', '05:')
    else:
        return date_str

# Función para convertir tiempo a segundos
def time_to_seconds(time_obj):
    return time_obj.hour * 3600 + time_obj.minute * 60 + time_obj.second

# Convertir la columna de tiempo a cadenas
acciones['HoraTeorIni'] = acciones['HoraTeorIni'].astype(str)
acciones['HoraTeorFin'] = acciones['HoraTeorFin'].astype(str)

# Aplicar la función para corregir los tiempos
acciones['HoraTeorIni'] = acciones['HoraTeorIni'].apply(fix_time)
acciones['HoraTeorFin'] = acciones['HoraTeorFin'].apply(fix_time)

# Convertir la columna de tiempo a datetime, usando errors='coerce' para manejar errores
acciones['HoraTeorIni'] = pd.to_datetime(acciones['HoraTeorIni'], format='%H:%M:%S', errors='coerce')
acciones['HoraTeorFin'] = pd.to_datetime(acciones['HoraTeorFin'], format='%H:%M:%S', errors='coerce')

# Eliminar la fecha predeterminada para trabajar solo con la parte de tiempo
acciones['HoraTeorIni'] = acciones['HoraTeorIni'].dt.time
acciones['HoraTeorFin'] = acciones['HoraTeorFin'].dt.time

# Manejar NaT después de la conversión
acciones['HoraTeorIni'] = acciones['HoraTeorIni'].apply(lambda x: x if pd.notnull(x) else pd.Timestamp('00:00:00').time())
acciones['HoraTeorFin'] = acciones['HoraTeorFin'].apply(lambda x: x if pd.notnull(x) else pd.Timestamp('00:00:00').time())

# Convertir la columna 'Instante' a segundos
acciones['HoraTeorIni_seg'] = acciones['HoraTeorIni'].apply(time_to_seconds).astype(int)
acciones['HoraTeorFin_seg'] = acciones['HoraTeorFin'].apply(time_to_seconds).astype(int)

# Convertir la columna 'Instante' a segundos
acciones

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Viaje Ini1,Viaje_fin,Viaje_fin1,HoraTeorIni,HoraFinTeor2,HoraFinTeor3,HoraFinTeor4,HoraTeorFin,HoraTeorIni_seg,HoraTeorFin_seg
0,2025-05-28,10:41:08,1091,40,Z50-4411,504411,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,1,2,2,10:34:18,10:39:18,12:01:00,12:06:00,12:06:00,38058,43560
1,2025-05-28,11:33:43,1091,27,Z50-4347,504347,5,Eliminar Coche,ECM,Esteban Camilo Medina,...,2,4,3,11:08:58,11:13:58,13:13:30,13:18:30,13:18:30,40138,47910
2,2025-05-28,12:39:57,1182,29,Z50-4485,504485,5,Eliminar Coche,GWM,William Munar Gonzalez,...,5,8,7,12:43:00,12:44:00,16:56:30,17:06:30,17:06:30,45780,61590
3,2025-05-28,13:01:24,1083,1,Z50-7077,507077,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,3,5,4,12:50:10,12:55:10,17:20:30,17:30:30,17:30:30,46210,63030
4,2025-05-28,13:32:34,1215,6,Z50-4259,504259,5,Eliminar Coche,BEJA,Jenny Alexandra Bernal,...,9,7,11,13:00:00,13:05:00,15:00:15,15:10:15,15:10:15,46800,54615
5,2025-05-28,4:37:02,1182,7,Z50-4185,504185,5,Eliminar Coche,dyrojas1,Deisi Yanira Rojas Torres,...,2,3,3,04:10:34,04:15:34,05:43:00,05:53:00,05:53:00,15034,21180
6,2025-05-28,6:34:18,1031,17,Z50-7118,507118,5,Eliminar Coche,VSC,Sandra Carolina Valeriano,...,2,4,4,06:06:23,06:07:23,10:28:30,10:29:30,10:05:30,21983,36330
7,2025-05-28,6:47:56,1103,21,Z50-2092,502092,5,Eliminar Coche,PAE,Edith Patarroyo,...,0,2,1,06:42:30,07:10:42,None,None,07:10:42,24150,25842
8,2025-05-28,6:49:58,1435,14,Z50-7008,507008,5,Eliminar Coche,NRN,Norberto Niño Rodriguez,...,0,3,2,06:46:00,08:03:30,08:08:30,None,08:08:30,24360,29310
9,2025-05-28,6:59:08,1062,17,Z50-4182,504182,5,Eliminar Coche,FLTC,Francy Linnie Timaran Cuesta,...,1,3,3,06:30:16,06:31:16,08:59:45,09:00:45,09:00:45,23416,32445


In [ ]:
acciones.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2024/Notas/{dia}_acciones_gen.csv', sep= ';', index=False)

Eliminación de servicio programado de IPH vs acciones

In [ ]:
# # Filtrar solo las filas donde hay exactamente dos cifras antes del primer ':'
# acciones= acciones[acciones['HoraTeorIni'].str.match(r'^\d{2}:\d{1,3}:\d{1,2}$')]

# # Mostrar el resultado
# acciones.head()

In [ ]:
# Inicializar la columna en iph1
iph1['Motivo_eliminacion'] = ''

# Iterar sobre acciones y asignar valores a iph1
for _, row in acciones.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['ServBus'] == row['Servicio_eliminado']) &
        (iph1['Instante_seg'] >= row['HoraTeorIni_seg']) &
        (iph1['Instante_seg'] <= row['HoraTeorFin_seg'])
    )
    iph1.loc[mask, 'Motivo_eliminacion'] = row['DescripcionMotivo']

# Ver resultado
iph1.head()

,JornadaTipo,TipoDia,Operador,Instante,ServBus,Evento,Linea,Coche,Sublinea,Ruta,Punto,TipoNodo,Viaje,ServicioCondEnt,TurnoEnt,OperadorEnt,ServicioCondSal,TipoVehiculo,Instante_seg,Motivo_eliminacion
0,GC250303T2,CN00111581,105,03:20:00,CNLRX0001,18,1300,1,NaN,NaN,142,5.0,1,CE130304,1.0,105.0,NaN,8,12000,
1,GC250303T2,CN00111581,105,04:00:00,CNLRX0001,4,1300,1,NaN,NaN,52372,1.0,1,NaN,NaN,NaN,NaN,8,14400,
2,GC250303T2,CN00111581,105,04:00:00,CNLRX0001,11,1300,1,4097.0,5658.0,52372,1.0,2,NaN,NaN,NaN,NaN,8,14400,
3,GC250303T2,CN00111581,105,04:08:48,CNLRX0001,0,1300,1,4097.0,5658.0,52802,1.0,2,NaN,NaN,NaN,NaN,8,14928,
4,GC250303T2,CN00111581,105,04:17:36,CNLRX0001,0,1300,1,4097.0,5658.0,53222,1.0,2,NaN,NaN,NaN,NaN,8,15456,


Llevar servicio de cambio de coche a IPH

In [ ]:
# Función para convertir HH:MM:SS a segundos
def tiempo_a_segundos(tiempo):
    h, m, s = map(int, tiempo.split(":"))
    return h * 3600 + m * 60 + s

# Aplicar la conversión en las tres columnas
acciones_rev5['HoraIniTeor_seg'] = acciones_rev5['HoraIniTeor'].apply(tiempo_a_segundos)
acciones_rev5['HoraFinTeor_seg'] = acciones_rev5['HoraFinTeor'].apply(tiempo_a_segundos)

acciones_rev5.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,HoraIniTeor,Viaje_fin,HoraFinTeor,Nombre Servicio,Sustituyente de Sustituyente,Servicio Planificado,Numero de sustituyente,N Viajes,HoraIniTeor_seg,HoraFinTeor_seg
299,28/05/2025,6:46:02,1594,4,Z50-4240,504240,36,Cambiar Coche,ECM,Esteban Camilo Medina,...,06:51:00,8,23:49:30,CEID0G004,No,CEID00004,1,5,24660,85770
348,28/05/2025,7:23:14,1103,9,Z50-2089,502089,36,Cambiar Coche,PAE,Edith Patarroyo,...,07:26:30,7,23:41:30,CEIH7G009,No,CEIH70009,1,4,26790,85290
365,28/05/2025,7:41:33,1182,22,Z50-4241,504241,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,...,07:36:00,10,22:28:45,CEIJWG015,No,CEIJW0015,1,6,27360,80925
366,28/05/2025,7:42:36,1300,8,Z50-7099,507099,36,Cambiar Coche,FLTC,Francy Linnie Timaran Cuesta,...,07:38:00,8,23:52:00,CNLRXG011,No,CNLRX0011,1,5,27480,85920
390,28/05/2025,7:53:02,1083,5,Z50-7063,507063,36,Cambiar Coche,BEJA,Jenny Alexandra Bernal,...,07:53:45,5,22:31:45,CEIKBG005,No,CEIKB0005,1,2,28425,81105


In [ ]:
# Inicializar las columnas en iph1
iph1['1_Servicio_cambiar_coche'] = ''
iph1['2_Servicio_cambiar_coche'] = ''
iph1['3_Servicio_cambiar_coche'] = ''

# PRIMERA ASIGNACIÓN
for _, row in acciones_rev5.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['ServBus'] == row['Servicio Planificado']) &
        (iph1['Instante_seg'] >= row['HoraIniTeor_seg']) &
        (iph1['Instante_seg'] <= row['HoraFinTeor_seg'])
    )
    iph1.loc[mask, '1_Servicio_cambiar_coche'] = row['Nombre Servicio']

# SEGUNDA ASIGNACIÓN (basado en '1_Servicio_cambiar_coche')
for _, row in acciones_rev5.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['1_Servicio_cambiar_coche'] == row['Servicio Planificado']) &
        (iph1['Instante_seg'] >= row['HoraIniTeor_seg']) &
        (iph1['Instante_seg'] <= row['HoraFinTeor_seg'])
    )
    iph1.loc[mask, '2_Servicio_cambiar_coche'] = row['Nombre Servicio']

# TERCERA ASIGNACIÓN (basado en '2_Servicio_cambiar_coche')
for _, row in acciones_rev5.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['2_Servicio_cambiar_coche'] == row['Servicio Planificado']) &
        (iph1['Instante_seg'] >= row['HoraIniTeor_seg']) &
        (iph1['Instante_seg'] <= row['HoraFinTeor_seg'])
    )
    iph1.loc[mask, '3_Servicio_cambiar_coche'] = row['Nombre Servicio']

# Ver resultado
iph1.head()

,JornadaTipo,TipoDia,Operador,Instante,ServBus,Evento,Linea,Coche,Sublinea,Ruta,...,ServicioCondEnt,TurnoEnt,OperadorEnt,ServicioCondSal,TipoVehiculo,Instante_seg,Motivo_eliminacion,1_Servicio_cambiar_coche,2_Servicio_cambiar_coche,3_Servicio_cambiar_coche
0,GC250303T2,CN00111581,105,03:20:00,CNLRX0001,18,1300,1,NaN,NaN,...,CE130304,1.0,105.0,NaN,8,12000,,,,
1,GC250303T2,CN00111581,105,04:00:00,CNLRX0001,4,1300,1,NaN,NaN,...,NaN,NaN,NaN,NaN,8,14400,,,,
2,GC250303T2,CN00111581,105,04:00:00,CNLRX0001,11,1300,1,4097.0,5658.0,...,NaN,NaN,NaN,NaN,8,14400,,,,
3,GC250303T2,CN00111581,105,04:08:48,CNLRX0001,0,1300,1,4097.0,5658.0,...,NaN,NaN,NaN,NaN,8,14928,,,,
4,GC250303T2,CN00111581,105,04:17:36,CNLRX0001,0,1300,1,4097.0,5658.0,...,NaN,NaN,NaN,NaN,8,15456,,,,


Eliminación de cambio de coche en IPH vs Acciones

In [ ]:
# Inicializar las columnas en iph1
iph1['Motivo_eliminacion_1'] = ''
iph1['Motivo_eliminacion_2'] = ''
iph1['Motivo_eliminacion_3'] = ''

# PRIMERA ASIGNACIÓN (Para '1_Servicio_cambiar_coche')
for _, row in acciones.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['1_Servicio_cambiar_coche'] == row['Servicio_eliminado']) &
        (iph1['Instante_seg'] >= row['HoraTeorIni_seg']) &
        (iph1['Instante_seg'] <= row['HoraTeorFin_seg'])
    )
    iph1.loc[mask, 'Motivo_eliminacion_1'] = row['DescripcionMotivo']

# SEGUNDA ASIGNACIÓN (Para '2_Servicio_cambiar_coche')
for _, row in acciones.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['2_Servicio_cambiar_coche'] == row['Servicio_eliminado']) &
        (iph1['Instante_seg'] >= row['HoraTeorIni_seg']) &
        (iph1['Instante_seg'] <= row['HoraTeorFin_seg'])
    )
    iph1.loc[mask, 'Motivo_eliminacion_2'] = row['DescripcionMotivo']

# TERCERA ASIGNACIÓN (Para '3_Servicio_cambiar_coche')
for _, row in acciones.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['3_Servicio_cambiar_coche'] == row['Servicio_eliminado']) &
        (iph1['Instante_seg'] >= row['HoraTeorIni_seg']) &
        (iph1['Instante_seg'] <= row['HoraTeorFin_seg'])
    )
    iph1.loc[mask, 'Motivo_eliminacion_3'] = row['DescripcionMotivo']

# Ver resultado
iph1.head()


,JornadaTipo,TipoDia,Operador,Instante,ServBus,Evento,Linea,Coche,Sublinea,Ruta,...,ServicioCondSal,TipoVehiculo,Instante_seg,Motivo_eliminacion,1_Servicio_cambiar_coche,2_Servicio_cambiar_coche,3_Servicio_cambiar_coche,Motivo_eliminacion_1,Motivo_eliminacion_2,Motivo_eliminacion_3
0,GC250303T2,CN00111581,105,03:20:00,CNLRX0001,18,1300,1,NaN,NaN,...,NaN,8,12000,,,,,,,
1,GC250303T2,CN00111581,105,04:00:00,CNLRX0001,4,1300,1,NaN,NaN,...,NaN,8,14400,,,,,,,
2,GC250303T2,CN00111581,105,04:00:00,CNLRX0001,11,1300,1,4097.0,5658.0,...,NaN,8,14400,,,,,,,
3,GC250303T2,CN00111581,105,04:08:48,CNLRX0001,0,1300,1,4097.0,5658.0,...,NaN,8,14928,,,,,,,
4,GC250303T2,CN00111581,105,04:17:36,CNLRX0001,0,1300,1,4097.0,5658.0,...,NaN,8,15456,,,,,,,


Llevar el servicio de introducir coche a IPH

In [ ]:
# Función para convertir HH:MM:SS a segundos
def tiempo_a_segundos(tiempo):
    h, m, s = map(int, tiempo.split(":"))
    return h * 3600 + m * 60 + s

# Aplicar la conversión en las tres columnas
acciones_rev4['HoraIniTeor_seg'] = acciones_rev4['HoraIniTeor'].apply(tiempo_a_segundos)
acciones_rev4['HoraFinTeor_seg'] = acciones_rev4['HoraFinTeor'].apply(tiempo_a_segundos)

acciones_rev4.head()

,Fecha,Instante,Linea,Coche,CodigoBus,NumBus,Accion,DescripcionAccion,Usuario,NombreUsuario,...,Viaje Ini1,Nombre Servicio,HoraIniTeor,Viaje_fin,HoraFinTeor,Viaje_fin1,Con Incorporacion,N Viajes,HoraIniTeor_seg,HoraFinTeor_seg
431,28/05/2025,8:16:03,1182,43,NaN,0,4,Introducir Coche,GWM,William Munar Gonzalez,...,3,RTB822503,08:06:00,4,10:07:00,3,1,1,29160,36420
629,28/05/2025,9:51:24,1031,33,NaN,0,4,Introducir Coche,VSC,Sandra Carolina Valeriano,...,3,RTA310803,10:14:45,4,12:56:45,3,1,1,36885,46605


In [ ]:
# Inicializar las columnas en iph1
iph1['1_servicio_introducir_coche'] = ''
iph1['2_servicio_introducir_coche'] = ''
iph1['3_servicio_introducir_coche'] = ''

# PRIMERA ASIGNACIÓN (Para '1_servicio_introducir_coche')
for _, row in acciones_rev4.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['ServBus'] == row['Servicio de Referencia']) &
        (iph1['Instante_seg'] >= row['HoraIniTeor_seg']) &
        (iph1['Instante_seg'] <= row['HoraFinTeor_seg'])
    )
    iph1.loc[mask, '1_servicio_introducir_coche'] = row['Nombre Servicio']

# SEGUNDA ASIGNACIÓN (Para '2_servicio_introducir_coche')
for _, row in acciones_rev4.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['1_servicio_introducir_coche'] == row['Servicio de Referencia']) &
        (iph1['Instante_seg'] >= row['HoraIniTeor_seg']) &
        (iph1['Instante_seg'] <= row['HoraFinTeor_seg'])
    )
    iph1.loc[mask, '2_servicio_introducir_coche'] = row['Nombre Servicio']

# TERCERA ASIGNACIÓN (Para '3_servicio_introducir_coche')
for _, row in acciones_rev4.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['2_servicio_introducir_coche'] == row['Servicio de Referencia']) &
        (iph1['Instante_seg'] >= row['HoraIniTeor_seg']) &
        (iph1['Instante_seg'] <= row['HoraFinTeor_seg'])
    )
    iph1.loc[mask, '3_servicio_introducir_coche'] = row['Nombre Servicio']

# Ver resultado
iph1.head()

,JornadaTipo,TipoDia,Operador,Instante,ServBus,Evento,Linea,Coche,Sublinea,Ruta,...,Motivo_eliminacion,1_Servicio_cambiar_coche,2_Servicio_cambiar_coche,3_Servicio_cambiar_coche,Motivo_eliminacion_1,Motivo_eliminacion_2,Motivo_eliminacion_3,1_servicio_introducir_coche,2_servicio_introducir_coche,3_servicio_introducir_coche
0,GC250303T2,CN00111581,105,03:20:00,CNLRX0001,18,1300,1,NaN,NaN,...,,,,,,,,,,
1,GC250303T2,CN00111581,105,04:00:00,CNLRX0001,4,1300,1,NaN,NaN,...,,,,,,,,,,
2,GC250303T2,CN00111581,105,04:00:00,CNLRX0001,11,1300,1,4097.0,5658.0,...,,,,,,,,,,
3,GC250303T2,CN00111581,105,04:08:48,CNLRX0001,0,1300,1,4097.0,5658.0,...,,,,,,,,,,
4,GC250303T2,CN00111581,105,04:17:36,CNLRX0001,0,1300,1,4097.0,5658.0,...,,,,,,,,,,


In [ ]:
# Inicializar las columnas en iph1
iph1['1_servicio_introducir_coche_servcambiar'] = ''
iph1['2_servicio_introducir_coche_servcambiar'] = ''
iph1['3_servicio_introducir_coche_servcambiar'] = ''

# PRIMERA ASIGNACIÓN (Para '1_servicio_introducir_coche_servcambiar')
for _, row in acciones_rev4.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['1_Servicio_cambiar_coche'] == row['Servicio de Referencia']) &
        (iph1['Instante_seg'] >= row['HoraIniTeor_seg']) &
        (iph1['Instante_seg'] <= row['HoraFinTeor_seg'])
    )
    iph1.loc[mask, '1_servicio_introducir_coche_servcambiar'] = row['Nombre Servicio']

# SEGUNDA ASIGNACIÓN (Para '2_servicio_introducir_coche_servcambiar')
for _, row in acciones_rev4.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['2_Servicio_cambiar_coche'] == row['Servicio de Referencia']) &
        (iph1['Instante_seg'] >= row['HoraIniTeor_seg']) &
        (iph1['Instante_seg'] <= row['HoraFinTeor_seg'])
    )
    iph1.loc[mask, '2_servicio_introducir_coche_servcambiar'] = row['Nombre Servicio']

# TERCERA ASIGNACIÓN (Para '3_servicio_introducir_coche_servcambiar')
for _, row in acciones_rev4.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['3_Servicio_cambiar_coche'] == row['Servicio de Referencia']) &
        (iph1['Instante_seg'] >= row['HoraIniTeor_seg']) &
        (iph1['Instante_seg'] <= row['HoraFinTeor_seg'])
    )
    iph1.loc[mask, '3_servicio_introducir_coche_servcambiar'] = row['Nombre Servicio']

# Ver resultado
iph1.head()

,JornadaTipo,TipoDia,Operador,Instante,ServBus,Evento,Linea,Coche,Sublinea,Ruta,...,3_Servicio_cambiar_coche,Motivo_eliminacion_1,Motivo_eliminacion_2,Motivo_eliminacion_3,1_servicio_introducir_coche,2_servicio_introducir_coche,3_servicio_introducir_coche,1_servicio_introducir_coche_servcambiar,2_servicio_introducir_coche_servcambiar,3_servicio_introducir_coche_servcambiar
0,GC250303T2,CN00111581,105,03:20:00,CNLRX0001,18,1300,1,NaN,NaN,...,,,,,,,,,,
1,GC250303T2,CN00111581,105,04:00:00,CNLRX0001,4,1300,1,NaN,NaN,...,,,,,,,,,,
2,GC250303T2,CN00111581,105,04:00:00,CNLRX0001,11,1300,1,4097.0,5658.0,...,,,,,,,,,,
3,GC250303T2,CN00111581,105,04:08:48,CNLRX0001,0,1300,1,4097.0,5658.0,...,,,,,,,,,,
4,GC250303T2,CN00111581,105,04:17:36,CNLRX0001,0,1300,1,4097.0,5658.0,...,,,,,,,,,,


In [ ]:
# Inicializar las columnas en iph1
iph1['1_Motivo_eliminacion_introducir_coche'] = ''
iph1['2_Motivo_eliminacion_introducir_coche'] = ''
iph1['3_Motivo_eliminacion_introducir_coche'] = ''

# PRIMERA ASIGNACIÓN (Para '1_Motivo_eliminacion_introducir_coche')
for _, row in acciones.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['1_servicio_introducir_coche'] == row['Servicio_eliminado']) &
        (iph1['Instante_seg'] >= row['HoraTeorIni_seg']) &
        (iph1['Instante_seg'] <= row['HoraTeorFin_seg'])
    )
    iph1.loc[mask, '1_Motivo_eliminacion_introducir_coche'] = row['DescripcionMotivo']

# SEGUNDA ASIGNACIÓN (Para '2_Motivo_eliminacion_introducir_coche')
for _, row in acciones.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['2_servicio_introducir_coche'] == row['Servicio_eliminado']) &
        (iph1['Instante_seg'] >= row['HoraTeorIni_seg']) &
        (iph1['Instante_seg'] <= row['HoraTeorFin_seg'])
    )
    iph1.loc[mask, '2_Motivo_eliminacion_introducir_coche'] = row['DescripcionMotivo']

# TERCERA ASIGNACIÓN (Para '3_Motivo_eliminacion_introducir_coche')
for _, row in acciones.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['3_servicio_introducir_coche'] == row['Servicio_eliminado']) &
        (iph1['Instante_seg'] >= row['HoraTeorIni_seg']) &
        (iph1['Instante_seg'] <= row['HoraTeorFin_seg'])
    )
    iph1.loc[mask, '3_Motivo_eliminacion_introducir_coche'] = row['DescripcionMotivo']

# Ver resultado
iph1.head()

,JornadaTipo,TipoDia,Operador,Instante,ServBus,Evento,Linea,Coche,Sublinea,Ruta,...,Motivo_eliminacion_3,1_servicio_introducir_coche,2_servicio_introducir_coche,3_servicio_introducir_coche,1_servicio_introducir_coche_servcambiar,2_servicio_introducir_coche_servcambiar,3_servicio_introducir_coche_servcambiar,1_Motivo_eliminacion_introducir_coche,2_Motivo_eliminacion_introducir_coche,3_Motivo_eliminacion_introducir_coche
0,GC250303T2,CN00111581,105,03:20:00,CNLRX0001,18,1300,1,NaN,NaN,...,,,,,,,,,,
1,GC250303T2,CN00111581,105,04:00:00,CNLRX0001,4,1300,1,NaN,NaN,...,,,,,,,,,,
2,GC250303T2,CN00111581,105,04:00:00,CNLRX0001,11,1300,1,4097.0,5658.0,...,,,,,,,,,,
3,GC250303T2,CN00111581,105,04:08:48,CNLRX0001,0,1300,1,4097.0,5658.0,...,,,,,,,,,,
4,GC250303T2,CN00111581,105,04:17:36,CNLRX0001,0,1300,1,4097.0,5658.0,...,,,,,,,,,,


In [ ]:
# Inicializar las columnas en iph1
iph1['1_Motivo_eliminacion_cambiar_coche_servcambiar'] = ''
iph1['2_Motivo_eliminacion_cambiar_coche_servcambiar'] = ''
iph1['3_Motivo_eliminacion_cambiar_coche_servcambiar'] = ''

# PRIMERA ASIGNACIÓN (Para '1_Motivo_eliminacion_cambiar_coche_servcambiar')
for _, row in acciones.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['1_servicio_introducir_coche_servcambiar'] == row['Servicio_eliminado']) &
        (iph1['Instante_seg'] >= row['HoraTeorIni_seg']) &
        (iph1['Instante_seg'] <= row['HoraTeorFin_seg'])
    )
    iph1.loc[mask, '1_Motivo_eliminacion_cambiar_coche_servcambiar'] = row['DescripcionMotivo']

# SEGUNDA ASIGNACIÓN (Para '2_Motivo_eliminacion_cambiar_coche_servcambiar')
for _, row in acciones.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['2_servicio_introducir_coche_servcambiar'] == row['Servicio_eliminado']) &
        (iph1['Instante_seg'] >= row['HoraTeorIni_seg']) &
        (iph1['Instante_seg'] <= row['HoraTeorFin_seg'])
    )
    iph1.loc[mask, '2_Motivo_eliminacion_cambiar_coche_servcambiar'] = row['DescripcionMotivo']

# TERCERA ASIGNACIÓN (Para '3_Motivo_eliminacion_cambiar_coche_servcambiar')
for _, row in acciones.iterrows():
    mask = (
        (iph1['Linea'] == row['Linea']) &
        (iph1['3_servicio_introducir_coche_servcambiar'] == row['Servicio_eliminado']) &
        (iph1['Instante_seg'] >= row['HoraTeorIni_seg']) &
        (iph1['Instante_seg'] <= row['HoraTeorFin_seg'])
    )
    iph1.loc[mask, '3_Motivo_eliminacion_cambiar_coche_servcambiar'] = row['DescripcionMotivo']

# Ver resultado
iph1.head()

,JornadaTipo,TipoDia,Operador,Instante,ServBus,Evento,Linea,Coche,Sublinea,Ruta,...,3_servicio_introducir_coche,1_servicio_introducir_coche_servcambiar,2_servicio_introducir_coche_servcambiar,3_servicio_introducir_coche_servcambiar,1_Motivo_eliminacion_introducir_coche,2_Motivo_eliminacion_introducir_coche,3_Motivo_eliminacion_introducir_coche,1_Motivo_eliminacion_cambiar_coche_servcambiar,2_Motivo_eliminacion_cambiar_coche_servcambiar,3_Motivo_eliminacion_cambiar_coche_servcambiar
0,GC250303T2,CN00111581,105,03:20:00,CNLRX0001,18,1300,1,NaN,NaN,...,,,,,,,,,,
1,GC250303T2,CN00111581,105,04:00:00,CNLRX0001,4,1300,1,NaN,NaN,...,,,,,,,,,,
2,GC250303T2,CN00111581,105,04:00:00,CNLRX0001,11,1300,1,4097.0,5658.0,...,,,,,,,,,,
3,GC250303T2,CN00111581,105,04:08:48,CNLRX0001,0,1300,1,4097.0,5658.0,...,,,,,,,,,,
4,GC250303T2,CN00111581,105,04:17:36,CNLRX0001,0,1300,1,4097.0,5658.0,...,,,,,,,,,,


In [ ]:
# Rellenar valores faltantes en Sublinea y Ruta basados en TipoDia, ServBus, y Coche
iph1[['Sublinea', 'Ruta']] = iph1.groupby(['TipoDia', 'ServBus', 'Coche'])[['Sublinea', 'Ruta']].transform(lambda x: x.ffill().bfill())

# Rellenar valores faltantes en ServicioCondEnt y ServicioCondSal basados en TipoDia, ServBus, Linea, Ruta, Coche y Viaje
iph1[['ServicioCondEnt', 'ServicioCondSal']] = iph1.groupby(['TipoDia', 'ServBus', 'Linea', 'Ruta', 'Coche'])[['ServicioCondEnt', 'ServicioCondSal']].transform(lambda x: x.ffill().bfill())

iph1

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9408\3751010593.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  iph1[['ServicioCondEnt', 'ServicioCondSal']] = iph1.groupby(['TipoDia', 'ServBus', 'Linea', 'Ruta', 'Coche'])[['ServicioCondEnt', 'ServicioCondSal']].transform(lambda x: x.ffill().bfill())


,JornadaTipo,TipoDia,Operador,Instante,ServBus,Evento,Linea,Coche,Sublinea,Ruta,...,3_servicio_introducir_coche,1_servicio_introducir_coche_servcambiar,2_servicio_introducir_coche_servcambiar,3_servicio_introducir_coche_servcambiar,1_Motivo_eliminacion_introducir_coche,2_Motivo_eliminacion_introducir_coche,3_Motivo_eliminacion_introducir_coche,1_Motivo_eliminacion_cambiar_coche_servcambiar,2_Motivo_eliminacion_cambiar_coche_servcambiar,3_Motivo_eliminacion_cambiar_coche_servcambiar
0,GC250303T2,CN00111581,105,03:20:00,CNLRX0001,18,1300,1,4097.0,5658.0,...,,,,,,,,,,
1,GC250303T2,CN00111581,105,04:00:00,CNLRX0001,4,1300,1,4097.0,5658.0,...,,,,,,,,,,
2,GC250303T2,CN00111581,105,04:00:00,CNLRX0001,11,1300,1,4097.0,5658.0,...,,,,,,,,,,
3,GC250303T2,CN00111581,105,04:08:48,CNLRX0001,0,1300,1,4097.0,5658.0,...,,,,,,,,,,
4,GC250303T2,CN00111581,105,04:17:36,CNLRX0001,0,1300,1,4097.0,5658.0,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38588,GMAD250519,CE00101255,105,20:58:45,CE4E70039,4,60,12,1665.0,2200.0,...,,,,,,,,,,
38589,GMAD250519,CE00101255,105,20:58:45,CE4E70039,3,60,12,1665.0,2200.0,...,,,,,,,,,,
38590,GMAD250519,CE00101255,105,21:40:15,CE4E70039,12,60,12,1665.0,2200.0,...,,,,,,,,,,
38591,GMAD250519,CE00101255,105,21:40:15,CE4E70039,2,60,12,1665.0,2200.0,...,,,,,,,,,,


In [ ]:
# Filtrar el DataFrame para que solo incluya filas donde 'Evento' sea 3 o 11
iph1= iph1[(iph1['Evento'] == 3) | (iph1['Evento'] == 11)]

iph1['Sublinea'] = iph1['Sublinea'].astype(int)
iph1['Ruta'] = iph1['Ruta'].astype(int)

#Eliminar columnas que no se necesitan para el proceso
col_eliminar = ['Operador', 'TurnoEnt', 'OperadorEnt']

iph1= iph1.drop(columns=col_eliminar)

#Completar columnas con 0

iph1['ServicioCondEnt'].fillna(0, inplace=True)
iph1['ServicioCondSal'].fillna(0, inplace=True)

iph1

C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9408\3724533145.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  iph1['Sublinea'] = iph1['Sublinea'].astype(int)
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9408\3724533145.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  iph1['Ruta'] = iph1['Ruta'].astype(int)
C:\Users\Jonny Villareal\AppData\Local\Temp\ipykernel_9408\3724533145.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignmen

,JornadaTipo,TipoDia,Instante,ServBus,Evento,Linea,Coche,Sublinea,Ruta,Punto,...,3_servicio_introducir_coche,1_servicio_introducir_coche_servcambiar,2_servicio_introducir_coche_servcambiar,3_servicio_introducir_coche_servcambiar,1_Motivo_eliminacion_introducir_coche,2_Motivo_eliminacion_introducir_coche,3_Motivo_eliminacion_introducir_coche,1_Motivo_eliminacion_cambiar_coche_servcambiar,2_Motivo_eliminacion_cambiar_coche_servcambiar,3_Motivo_eliminacion_cambiar_coche_servcambiar
2,GC250303T2,CN00111581,04:00:00,CNLRX0001,11,1300,1,4097,5658,52372,...,,,,,,,,,,
15,GC250303T2,CN00111581,05:52:00,CNLRX0001,3,1300,1,4097,5658,52372,...,,,,,,,,,,
35,GC250303T2,CN00111581,08:38:00,CNLRX0001,3,1300,1,4097,5658,52372,...,,,,,,,,,,
41,GC250303T2,CN00111581,11:02:00,CNLRX0001,3,1300,1,4097,5658,52372,...,,,,,,,,,,
54,GC250303T2,CN00111581,14:14:00,CNLRX0001,3,1300,1,4097,5658,52372,...,,,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38581,GMAD250519,CE00101255,16:53:00,CE4E70039,3,60,12,1665,2200,82,...,,,,,,,,,,
38583,GMAD250519,CE00101255,18:05:45,CE4E70039,3,60,12,1665,2200,82,...,,,,,,,,,,
38585,GMAD250519,CE00101255,19:15:45,CE4E70039,3,60,12,1665,2200,82,...,,,,,,,,,,
38587,GMAD250519,CE00101255,20:13:00,CE4E70039,3,60,12,1665,2200,82,...,,,,,,,,,,


In [ ]:
# Ordenar el DataFrame por 'Instante_seg', 'Linea', 'Ruta' y 'ServBus'
iph1 = iph1.sort_values(by=['Instante_seg', 'Linea', 'Ruta', 'ServBus']).reset_index(drop=True)

# Ver los primeros registros ordenados
iph1.head()

,JornadaTipo,TipoDia,Instante,ServBus,Evento,Linea,Coche,Sublinea,Ruta,Punto,...,3_servicio_introducir_coche,1_servicio_introducir_coche_servcambiar,2_servicio_introducir_coche_servcambiar,3_servicio_introducir_coche_servcambiar,1_Motivo_eliminacion_introducir_coche,2_Motivo_eliminacion_introducir_coche,3_Motivo_eliminacion_introducir_coche,1_Motivo_eliminacion_cambiar_coche_servcambiar,2_Motivo_eliminacion_cambiar_coche_servcambiar,3_Motivo_eliminacion_cambiar_coche_servcambiar
0,GM250526_V_FMS,CE0010007908,00:00:00,CE04F0004,3,10305,4,486,10590,52845,...,,,,,,,,,,
1,GEAH240902,CO00102510,00:11:30,CO9CE0005,3,172,1,658,884,11818,...,,,,,,,,,,
2,GEAH240902,CO00102510,00:12:15,CO9CE0002,3,173,1,337,467,11819,...,,,,,,,,,,
3,GMAD250519,CE00101255,00:15:00,CE4E70034,11,59,5,1975,2632,80,...,,,,,,,,,,
4,GMAD250519,CE00101255,00:15:00,CE4E70010,11,60,5,1664,2202,82,...,,,,,,,,,,


In [ ]:
#exportar desglosado
iph.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2024/Notas/{dia}_iph.csv', sep= ';', index=False)

In [ ]:
#exportar acciones
acciones.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2024/Notas/{dia}_acciones.csv', sep= ';', index=False)

In [ ]:
#exportar data
acciones_rev4.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2024/Notas/{dia}_acciones_introducir.csv', sep= ';', index=False)

In [ ]:
#exportar data
acciones_rev5.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2024/Notas/{dia}_acciones_cambiar.csv', sep= ';', index=False)

In [ ]:
#exportar data
iph1.to_csv(f'C:/Users/Jonny Villareal/OneDrive - Gmovil SAS/Escritorio/Jonny/Control 2024/Notas/{dia}_iph_bruto.csv', sep= ';', index=False)